# 1 - Data cleaning, feature engineering and model training

**Standalone Google Colab notebook.** Upload this notebook, select a CPU runtime and use **Runtime > Run all**. Choose your original case-study CSV when prompted. Everything needed is embedded below.

This notebook audits anomalies, isolates unlabeled rows, builds fold-local features and trains full-size XGBoost and Random Forest candidates with nested account-grouped calibration. It also evaluates the fixed ensemble and simple baselines. The final cell downloads **CLV_TRAINING_ARTIFACTS.zip**. Save it for Notebook 2. Fresh training is performed from your CSV. A separately labeled frozen model/evidence payload is included only to reproduce the published report in Notebook 2; it is not used for new model fitting.

Reference: 2026-10-01; seed: 42; outer/inner folds: 5/3; XGB/RF trees: 150/200; fixed raw mixture: 0.6/0.4. The twelve-month array label is not commercial account cancellation.

## 1. Prepare an isolated compute environment

Run all cells in order on a **CPU runtime**. Package installation is isolated from Colab; no kernel restart is needed. Python 3.12 is used. If the host uses another Python version, setup downloads an isolated 3.12.10 interpreter automatically. The first setup needs internet access. Exact scientific-library pins, seeds, folds, weights and reference date match the original implementation.

In [ ]:
import os, sys, json, hashlib, subprocess, venv, base64, io, zipfile, importlib.util
from pathlib import Path
from IPython.display import display, Markdown

PROJECT = Path(os.environ.get('CLV_COLLAB_WORKSPACE',
    '/content/clv_collab' if Path('/content').is_dir() else 'clv_collab_run')).resolve()
PROJECT.mkdir(parents=True, exist_ok=True)
for folder in ['src','sql','data','environment','outputs/cleaning','reference']:
    (PROJECT/folder).mkdir(parents=True,exist_ok=True)

def write_embedded(relative_path, text, crlf_lines, expected_sha):
    lines=text.splitlines(keepends=True)
    content=''.join(line[:-1]+'\r\n' if i in crlf_lines and line.endswith('\n') else line
                    for i,line in enumerate(lines)).encode('utf-8')
    if hashlib.sha256(content).hexdigest()!=expected_sha:
        raise AssertionError('Embedded source integrity failed: '+relative_path)
    (PROJECT/relative_path).write_bytes(content)

REQUIREMENTS = 'numpy==2.5.3\npandas==3.0.6\nscikit-learn==1.9.1\nxgboost==3.4.1\nshap==0.53.0\nmatplotlib==3.11.2\ntabulate==0.10.0\nscipy==1.18.1\njoblib==1.6.0\nnarwhals==2.26.0\nthreadpoolctl==3.7.0\ncloudpickle==3.1.2\nslicer==0.0.8\ntqdm==4.70.1\npackaging==26.3\ncontourpy==1.4.0\ncycler==0.12.1\nfonttools==4.66.1\nkiwisolver==1.5.1\npillow==12.3.0\npyparsing==3.3.3\npython-dateutil==2.9.0.post0\nsix==1.17.0\ntzdata==2026.5\n'
(PROJECT/'environment/requirements-colab.txt').write_bytes(REQUIREMENTS.encode())
SUPPORT_CODE = '"""Notebook-local runtime, artifact transfer and numerical parity checks."""\nimport csv\nimport hashlib\nimport html\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport subprocess\nimport sys\nimport textwrap\nimport zipfile\n\nPROJECT = Path(__file__).resolve().parent\n\ndef run_step(code, python):\n    env = os.environ.copy()\n    env.update(PYTHONHASHSEED=\'42\', OMP_NUM_THREADS=\'2\', OPENBLAS_NUM_THREADS=\'2\',\n               MKL_NUM_THREADS=\'2\', MPLBACKEND=\'Agg\', PYTHONDONTWRITEBYTECODE=\'1\',\n               PYTHONPATH=str(PROJECT))\n    # Separate interpreters keep Colab\'s imported/preinstalled packages isolated.\n    subprocess.run([str(python), \'-u\', \'-c\', textwrap.dedent(code)], cwd=PROJECT,\n                   env=env, check=True)\n\ndef show_csv(path, rows=25):\n    from IPython.display import display, HTML\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        reader = csv.reader(f)\n        header = next(reader)\n        records = []\n        for i, row in enumerate(reader):\n            if i >= rows: break\n            records.append(row)\n    def cells(row, tag):\n        return \'\'.join(f\'<{tag}>{html.escape(v)}</{tag}>\' for v in row)\n    body = \'<tr>\' + cells(header, \'th\') + \'</tr>\'\n    body += \'\'.join(\'<tr>\' + cells(row, \'td\') + \'</tr>\' for row in records)\n    display(HTML(\'<div style="overflow-x:auto"><table>\' + body + \'</table></div>\'))\n\ndef show_images(folder, names):\n    from IPython.display import display, Image\n    for name in names:\n        print(name)\n        display(Image(filename=str(Path(folder)/name), width=950))\n\ndef offer_download(path):\n    if os.environ.get(\'CLV_COLLAB_AUTOMATED\') == \'1\':\n        print(\'Download ready:\', path)\n        return\n    try:\n        from google.colab import files\n    except ImportError:\n        print(\'Download ready:\', path)\n    else:\n        files.download(str(path))\n\ndef canonical_input_sha(raw):\n    columns = [\'account_id\', \'array_id\', \'contract_end_date\', \'historical_15yr_net_profit_usd\',\n               \'current_capacity_tb\', \'utilization_rate_pct\', \'static_baseline_churn_rate\',\n               \'churn_event_next_12m\', \'expansion_potential_signal\']\n    if set(columns) - set(raw): return None\n    content = raw[columns].to_csv(index=False, date_format=\'%Y-%m-%d\', lineterminator=\'\\n\')\n    return hashlib.sha256(content.encode()).hexdigest()\n\ndef compare_tables(actual, expected, keys, ignore=(), atol=2e-6):\n    import numpy as np\n    import pandas as pd\n    if set(actual.columns) != set(expected.columns):\n        raise AssertionError(\'Reference column set differs\')\n    a = actual.set_index(keys).sort_index()\n    b = expected.set_index(keys).sort_index()\n    if not a.index.equals(b.index): raise AssertionError(\'Reference row keys differ\')\n    results = []\n    for c in a.columns:\n        if c in ignore: continue\n        x, y = a[c], b[c]\n        if pd.api.types.is_numeric_dtype(y) and not pd.api.types.is_bool_dtype(y):\n            x, y = x.to_numpy(dtype=float), y.to_numpy(dtype=float)\n            delta = float(np.nanmax(np.abs(x-y))) if np.isfinite(y).any() else 0.0\n            matched = np.allclose(x, y, atol=atol, rtol=0, equal_nan=True)\n        else:\n            delta = None\n            matched = x.fillna(\'<MISSING>\').astype(str).equals(y.fillna(\'<MISSING>\').astype(str))\n        results.append({\'column\': c, \'max_absolute_difference\': delta,\n                        \'absolute_tolerance\': atol if delta is not None else None, \'passed\': bool(matched)})\n    return results\n\ndef check_reference(stage, run_dir, score_dir=None, bundle_run_dir=None):\n    import pandas as pd\n    from src.config import DATA_PATH\n    from src.data_cleaning import validate_raw_frame\n    from src.model_training import sha256_file\n    from src.explainability import xgb_contributions\n    reference = PROJECT/\'reference\'\n    meta = json.loads((reference/\'reference.json\').read_text())\n    matched_input = canonical_input_sha(validate_raw_frame(DATA_PATH)) == meta[\'canonical_input_sha256\']\n    report = {\'stage\': stage, \'reference_revision\': meta[\'source_revision\'],\n              \'same_case_study_data\': matched_input, \'passed\': None, \'checks\': {}}\n    out = PROJECT/\'outputs\'/f\'reproducibility_{stage}.json\'\n    if not matched_input:\n        report[\'status\'] = \'NOT_APPLICABLE_DIFFERENT_INPUT\'\n        out.write_text(json.dumps(report, indent=2))\n        print(\'New/differently ordered data: published-output parity is not applicable.\')\n        return report\n    specs = [(\'evaluation_predictions.csv\', [\'account_id\',\'array_id\'], [\'run_id\'], 2e-6)]\n    if stage in (\'analysis\',\'published\'):\n        specs += [(\'metrics.csv\',[\'model\'],[],2e-6),\n                  (\'bootstrap_intervals.csv\',[\'model\',\'baseline\'],[],2e-6),\n                  (\'cohorts.csv\',[\'dimension\',\'cohort\'],[],2e-6),\n                  (\'historical_exposure_buckets.csv\',[\'risk_bucket\'],[],2e-6)]\n    for name, keys, ignore, tolerance in specs:\n        a, b = pd.read_csv(Path(run_dir)/name), pd.read_csv(reference/name)\n        report[\'checks\'][name] = compare_tables(a,b,keys,ignore,tolerance)\n    if stage in (\'analysis\',\'published\'):\n        score_dir = Path(score_dir)\n        report[\'checks\'][\'CLV_SCORED_OUTPUT.csv\'] = compare_tables(\n            pd.read_csv(score_dir/\'CLV_SCORED_OUTPUT.csv\'),pd.read_csv(reference/\'CLV_SCORED_OUTPUT.csv\'),\n            [\'account_id\',\'array_id\'], [\'scored_at\'],2e-6)\n        from src.model_training import load_bundle\n        bundle, _ = load_bundle(bundle_run_dir or run_dir)\n        raw = validate_raw_frame(DATA_PATH)\n        values, base, _ = xgb_contributions(bundle, raw)\n        actual = raw[[\'account_id\',\'array_id\']].copy()\n        actual[\'base_value\'] = base\n        for i, name in enumerate(bundle[\'preprocessor\'].feature_names_): actual[name] = values[:,i]\n        report[\'checks\'][\'full_xgb_contributions.csv\'] = compare_tables(\n            actual,pd.read_csv(reference/\'full_xgb_contributions.csv\'),[\'account_id\',\'array_id\'],atol=1e-5)\n        for path in sorted((reference/\'submission_analytics\').glob(\'*.csv\')):\n            name = path.name\n            keys = ([\'account_id\',\'scenario\',\'discount_rate\',\'array_id\'] if name==\'discounted_cash_flow_scenarios.csv\'\n                    else [\'account_id\',\'array_id\',\'scenario\'] if name==\'fifteen_year_scenarios.csv\'\n                    else [\'account_id\',\'array_id\'] if name==\'capacity_expansion_signals.csv\' else [\'account_id\'])\n            report[\'checks\'][name] = compare_tables(\n                pd.read_csv(score_dir/\'submission_analytics\'/name),pd.read_csv(path),keys,atol=1e-5)\n    report[\'passed\'] = all(c[\'passed\'] for group in report[\'checks\'].values() for c in group)\n    report[\'status\'] = (\'PASS_WITHIN_DOCUMENTED_TOLERANCES\' if report[\'passed\'] else\n                        \'FRESH_RETRAINING_DIFFERS_FROM_PUBLISHED\' if stage!=\'published\' else \'FAIL\')\n    report[\'reference_note\'] = \'Run IDs, UTC times, file/pickle hashes and plot bytes are not numerical parity targets.\'\n    out.write_text(json.dumps(report, indent=2))\n    for name, group in report[\'checks\'].items():\n        failed = [c[\'column\'] for c in group if not c[\'passed\']]\n        label=\'FAIL\' if failed and stage==\'published\' else \'DIFF\' if failed else \'PASS\'\n        print(label + \' \' + name + (\': \'+str(failed) if failed else \'\'))\n    if not report[\'passed\'] and stage==\'published\':\n        raise AssertionError(f\'Published-model replay parity failed; inspect {out}\')\n    if not report[\'passed\']:\n        print(\'Fresh retraining differs from the Windows reference. These results remain separate;\'\n              \' the published-model replay must pass its own strict numerical checks.\')\n    return report\n\ndef activate_published_run():\n    from src.config import DATA_PATH,OUTPUTS\n    from src.data_cleaning import validate_raw_frame\n    reference=json.loads((PROJECT/\'reference/reference.json\').read_text())\n    if canonical_input_sha(validate_raw_frame(DATA_PATH))!=reference[\'canonical_input_sha256\']:\n        (OUTPUTS/\'published_run.json\').write_text(json.dumps({\'run_id\':None}))\n        print(\'Published replay skipped: uploaded data differs from the frozen case study.\')\n        return None\n    run=OUTPUTS/\'runs\'/reference[\'training_run_id\']\n    run.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(PROJECT/\'published_training.zip\') as z:\n        for name in z.namelist():\n            if \'/\' in name or \'\\\\\' in name or name.startswith(\'.\'): raise ValueError(\'Unexpected frozen artifact path\')\n            content=z.read(name); target=run/name\n            if target.exists() and target.read_bytes()!=content: raise ValueError(\'Frozen published artifact changed: \'+name)\n            target.write_bytes(content)\n    from src.model_training import sha256_file,load_bundle\n    from src.backtesting import read_evidence\n    manifest=json.loads((run/\'manifest.json\').read_text())\n    assert sha256_file(run/\'training_snapshot.csv\')==manifest[\'training_snapshot_sha256\']\n    read_evidence(run); load_bundle(run)\n    (OUTPUTS/\'published_run.json\').write_text(json.dumps({\'run_id\':run.name}))\n    print(\'Published training artifacts verified:\',run.name,\'(replay, not new training)\')\n    return run\n\ndef make_archive(path, kind, run_dir, score_dir=None):\n    path = Path(path)\n    candidates = [PROJECT/\'data/Case_Study_2_CLV_Predictive_Analytics.csv\',\n                  PROJECT/\'environment/requirements-colab.txt\', PROJECT/\'outputs/latest_run.json\']\n    for folder in [PROJECT/\'src\', PROJECT/\'sql\', PROJECT/\'outputs/cleaning\', Path(run_dir)]:\n        candidates.extend(p for p in folder.rglob(\'*\') if p.is_file() and \'__pycache__\' not in p.parts)\n    candidates.extend((PROJECT/\'outputs\').glob(\'reproducibility_*.json\'))\n    published_pointer=PROJECT/\'outputs/published_run.json\'\n    if published_pointer.exists():\n        candidates.append(published_pointer)\n        published_id=json.loads(published_pointer.read_text())[\'run_id\']\n        if published_id:\n            candidates.extend(p for p in (PROJECT/\'outputs/runs\'/published_id).glob(\'*\') if p.is_file())\n    if score_dir is not None:\n        candidates.append(PROJECT/\'outputs/latest_score_run.json\')\n        candidates.append(PROJECT/\'outputs/backtesting_report.md\')\n        candidates.extend(p for p in Path(score_dir).rglob(\'*\') if p.is_file())\n        candidates.extend(p for p in (PROJECT/\'outputs/shap_plots\').glob(\'*.png\'))\n        candidates.extend(p for p in (PROJECT/\'outputs/analysis_plots\').glob(\'*.png\'))\n        candidates.append(PROJECT/\'outputs/snowflake_handoff.sql\')\n        for folder in [\'published_validation\',\'published_scores\',\'published_shap_plots\']:\n            candidates.extend(p for p in (PROJECT/\'outputs\'/folder).rglob(\'*\') if p.is_file())\n        handoff=PROJECT/\'outputs/published_snowflake_handoff.sql\'\n        if handoff.exists(): candidates.append(handoff)\n        pointer=PROJECT/\'outputs/latest_published_score_run.json\'\n        if pointer.exists(): candidates.append(pointer)\n    files = {p.relative_to(PROJECT).as_posix(): p for p in candidates}\n    metadata = {\'format_version\':1,\'kind\':kind,\'training_run_id\':Path(run_dir).name,\n                \'files\':{n:{\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest(),\'bytes\':p.stat().st_size}\n                         for n,p in sorted(files.items())}}\n    with zipfile.ZipFile(path,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for name,p in sorted(files.items()): z.write(p,name)\n        z.writestr(\'archive_manifest.json\',json.dumps(metadata,indent=2))\n    print(f\'{kind} archive: {path.name}; {len(files)} files; {path.stat().st_size:,} bytes\')\n    return path\n\ndef restore_training_archive(path):\n    # Transfer only an archive generated by Notebook 1; check bytes before loading pickle.\n    with zipfile.ZipFile(path) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)): raise ValueError(\'Duplicate archive paths\')\n        meta=json.loads(z.read(\'archive_manifest.json\'))\n        if meta.get(\'format_version\')!=1 or meta.get(\'kind\')!=\'clv_training\':\n            raise ValueError(\'Upload the CLV_TRAINING_ARTIFACTS.zip produced by Notebook 1\')\n        if set(names)!=set(meta[\'files\'])|{\'archive_manifest.json\'}:\n            raise ValueError(\'Archive manifest/member mismatch\')\n        if sum(i.file_size for i in z.infolist())>100_000_000: raise ValueError(\'Archive is unexpectedly large\')\n        checked=[]\n        for name,info in meta[\'files\'].items():\n            p=PurePosixPath(name)\n            if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or p.parts[0] not in {\'src\',\'sql\',\'data\',\'environment\',\'outputs\'}:\n                raise ValueError(\'Invalid archive path\')\n            content=z.read(name)\n            if hashlib.sha256(content).hexdigest()!=info[\'sha256\'] or len(content)!=info[\'bytes\']:\n                raise ValueError(\'Archive integrity mismatch: \'+name)\n            target=PROJECT.joinpath(*p.parts)\n            if p.parts[0] in {\'src\',\'sql\',\'environment\'} and target.read_bytes()!=content:\n                raise ValueError(\'Notebook/source/environment differs from the training archive: \'+name)\n            checked.append((target,content))\n        for target,content in checked:\n            if target.exists() and target.read_bytes()!=content:\n                raise ValueError(\'Existing workspace differs; use a fresh runtime: \'+str(target))\n        for target,content in checked:\n            target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(content)\n    print(\'Restored verified training archive:\',meta[\'training_run_id\'])\n    return meta\n'
(PROJECT/'collab_support.py').write_bytes(SUPPORT_CODE.encode())
spec=importlib.util.spec_from_file_location('collab_support', PROJECT/'collab_support.py')
support=importlib.util.module_from_spec(spec)
sys.modules['collab_support']=support; spec.loader.exec_module(support)
from collab_support import show_csv, show_images, offer_download

override=os.environ.get('CLV_COLLAB_PYTHON')  # Automated verification only; leave unset in Colab.
COMPUTE_PYTHON=Path(override) if override else PROJECT/'.venv'/('Scripts/python.exe' if os.name=='nt' else 'bin/python')
if not override:
    if not COMPUTE_PYTHON.exists():
        if sys.version_info[:2]==(3,12):
            venv.EnvBuilder(with_pip=False).create(PROJECT/'.venv')
        else:
            # Bootstrap an isolated Python 3.12 when Colab's default Python changes.
            bootstrap=PROJECT/'.bootstrap'
            subprocess.run([sys.executable,'-m','pip','install','--quiet','--target',str(bootstrap),'uv==0.8.22'],check=True)
            bootstrap_env=os.environ.copy(); bootstrap_env['PYTHONPATH']=str(bootstrap)
            subprocess.run([sys.executable,'-m','uv','venv','--python','3.12.10','--python-preference','only-managed','--seed',str(PROJECT/'.venv')],
                           env=bootstrap_env,check=True)
    subprocess.run([sys.executable,'-m','pip','--python',str(COMPUTE_PYTHON),'install',
                    '--quiet','--only-binary=:all:','-r',str(PROJECT/'environment/requirements-colab.txt')],check=True)
    subprocess.run([sys.executable,'-m','pip','--python',str(COMPUTE_PYTHON),'check'],check=True)

def run_step(code): support.run_step(code, COMPUTE_PYTHON)
run_step("""
import importlib.metadata as md, json, platform
from pathlib import Path
versions = {}
for line in Path('environment/requirements-colab.txt').read_text().splitlines():
    name, expected = line.split('==')
    actual = md.version(name)
    assert actual == expected, f'{name}: expected {expected}, found {actual}'
    versions[name] = actual
assert platform.python_version_tuple()[:2] == ('3','12'), 'Use the isolated Python 3.12 interpreter'
record = {'python':platform.python_version(),'platform':platform.platform(),'packages':versions,
          'all_distributions':{d.metadata['Name']:d.version for d in md.distributions()}}
Path('environment/runtime.json').write_text(json.dumps(record,indent=2))
print('Isolated compute environment:',record['python'],record['platform'])
""")
print('Workspace:', PROJECT)


## 2. Embedded implementation

These cells materialize the complete Python modules and SQL inside the runtime. No GitHub clone, repository upload or external `.py` files are required. Definitions live in stable modules so the saved model bundle can be loaded by the second notebook. Edit these only if deliberately changing the analysis; integrity checks protect the frozen version.

In [ ]:
# Embedded src/__init__.py
MODULE_CODE = r'''# CLV Predictive Analytics — Source Package
'''
write_embedded('src/__init__.py', MODULE_CODE, [], '01fd1217d01520f633a4d3528abc20dacb128d3905ab3ab41768d7161a944a76')

In [ ]:
# Embedded src/backtesting.py
MODULE_CODE = r'''"""Read immutable grouped OOF evidence; generate result-dependent evaluation."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from src.config import OUTPUTS, PRIMARY_SCORE
from src.data_cleaning import TARGET
from src.model_training import latest_run, sha256_file
from src.utils import assign_risk_bucket


def score_columns(df):
    return [c for c in df if c in ('static_5', 'training_prevalence', 'logistic')
            or c.startswith(('weighted_', 'unweighted_'))]


def validate_evaluation(df, manifest):
    required = {'account_id', 'array_id', TARGET, 'fold_id', 'run_id', 'prediction_source'}
    if required - set(df):
        raise ValueError('Evaluation requires keyed OOF evidence, fold IDs and lineage')
    if manifest.get('prediction_source') != 'grouped_oof' or not df.prediction_source.eq('grouped_oof').all():
        raise ValueError('Fitted-production scores cannot be used for evaluation')
    if not df.run_id.eq(manifest['run_id']).all() or len(df) != manifest['labeled_rows']:
        raise ValueError('Evaluation run/row-count mismatch')
    if df.duplicated(['account_id', 'array_id']).any() or not df[TARGET].isin([0, 1]).all():
        raise ValueError('Duplicate keys or invalid/missing evaluation labels')
    if df.groupby('account_id').fold_id.nunique().max() != 1:
        raise ValueError('Sibling arrays must stay in the same validation fold')
    all_accounts = set(df.account_id)
    ids = set()
    for fold in manifest['folds']:
        fid = fold['fold_id']
        if fid in ids:
            raise ValueError('Duplicate fold ID in manifest')
        ids.add(fid)
        tr, va = set(fold['train_accounts']), set(fold['validation_accounts'])
        actual = df.loc[df.fold_id.eq(fid)]
        if tr & va or tr != all_accounts - va or va != set(actual.account_id):
            raise ValueError('Training/validation account isolation failed')
        if len(actual) != fold['validation_rows']:
            raise ValueError('Fold row-count mismatch')
        if fold['train_rows'] != len(df) - len(actual):
            raise ValueError('Training row-count mismatch')
        expected_prevalence = df.loc[~df.fold_id.eq(fid), TARGET].mean()
        if not np.isclose(expected_prevalence, fold['training_prevalence']):
            raise ValueError('Prevalence baseline must use training labels only')
        if not np.allclose(actual.training_prevalence, fold['training_prevalence']):
            raise ValueError('Fold prevalence mismatch')
    if set(df.fold_id) != ids:
        raise ValueError('Unknown or missing fold assignment')
    cols = score_columns(df)
    if PRIMARY_SCORE not in cols or not {'static_5', 'training_prevalence', 'logistic'} <= set(cols):
        raise ValueError('Missing candidate or baseline probability columns')
    p = df[cols].to_numpy(dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Evaluation probabilities must be finite in [0, 1]')


def read_evidence(run_dir=None):
    run_dir = Path(run_dir) if run_dir is not None else latest_run()
    manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
    file = run_dir / 'evaluation_predictions.csv'
    if sha256_file(file) != manifest['evaluation_sha256']:
        raise ValueError('OOF evidence hash mismatch')
    df = pd.read_csv(file, dtype={'account_id': str, 'array_id': str, 'run_id': str})
    validate_evaluation(df, manifest)
    return df, manifest


def expected_calibration_error(y, p, quantile=False):
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 11))) if quantile else np.linspace(0, 1, 11)
    bins = np.searchsorted(edges[1:-1], p, side='right')
    return float(sum(np.mean(bins == k) * abs(np.mean(p[bins == k]) - np.mean(y[bins == k]))
                     for k in np.unique(bins)))


def probability_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {
        'mean_probability': float(p.mean()), 'observed_rate': float(y.mean()),
        'bias': float(p.mean() - y.mean()), 'brier': float(np.mean((p - y) ** 2)),
        'log_loss': float(log_loss(y, np.clip(p, 1e-12, 1 - 1e-12), labels=[0, 1])),
        'roc_auc': float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None,
        'average_precision': float(average_precision_score(y, p)) if y.sum() else None,
        'ece_equal_width': expected_calibration_error(y, p),
        'ece_equal_count': expected_calibration_error(y, p, quantile=True),
    }


def paired_account_ci(df, candidate, baseline, repetitions=1000, seed=20261011):
    """Paired fixed-prediction percentile CI; whole accounts are resampled."""
    y = df[TARGET].to_numpy()
    p, b = df[candidate].to_numpy(), df[baseline].to_numpy()
    values = pd.DataFrame({
        'account_id': df.account_id, 'n': 1,
        'gain': (b - y) ** 2 - (p - y) ** 2, 'bias': p - y,
    }).groupby('account_id').sum()
    rng = np.random.default_rng(seed)
    samples = rng.multinomial(len(values), np.full(len(values), 1 / len(values)), size=repetitions)
    n = samples @ values.n.to_numpy()
    gains = (samples @ values.gain.to_numpy()) / n
    biases = (samples @ values.bias.to_numpy()) / n
    return {
        'brier_gain': float(values.gain.sum() / values.n.sum()),
        'gain_ci_low': float(np.quantile(gains, .025)),
        'gain_ci_high': float(np.quantile(gains, .975)),
        'bias_ci_low': float(np.quantile(biases, .025)),
        'bias_ci_high': float(np.quantile(biases, .975)),
    }


def improvement_text(lift_pct):
    if abs(lift_pct) < 1e-10:
        return 'Brier score is unchanged relative to the baseline.'
    if lift_pct > 0:
        return f'Brier score improves by {lift_pct:.2f}% relative to the baseline.'
    return f'Brier score deteriorates by {-lift_pct:.2f}% relative to the baseline.'


def write_once(path, content):
    """Replaying a report preserves bytes; changed results require a new run."""
    path = Path(path)
    content = content.replace('\r\n', '\n')
    if path.exists():
        if path.read_text(encoding='utf-8') != content:
            raise ValueError('Evaluation artifacts are immutable; use a new run for changed results')
        return
    path.write_text(content, encoding='utf-8')


def run_backtesting_suite(evaluation, manifest, output_dir, repetitions=1000):
    """Only validated keyed grouped-OOF evidence is accepted (retired array API removed)."""
    validate_evaluation(evaluation, manifest)
    df = evaluation.copy()
    y = df[TARGET].to_numpy()
    metrics = pd.DataFrame([
        {'model': c, **probability_metrics(y, df[c].to_numpy())} for c in score_columns(df)
    ])
    base_brier = float(metrics.loc[metrics.model.eq('static_5'), 'brier'].iloc[0])
    metrics['brier_lift_vs_5_pct'] = 100 * (base_brier - metrics.brier) / base_brier
    ci = pd.DataFrame([
        {'model': candidate, 'baseline': baseline,
         **paired_account_ci(df, candidate, baseline, repetitions=repetitions)}
        for candidate in ['weighted_ensemble_raw', 'unweighted_xgb_raw', 'unweighted_xgb_sigmoid',
                          'unweighted_rf_raw', 'unweighted_rf_sigmoid', 'unweighted_ensemble_raw',
                          PRIMARY_SCORE, 'training_prevalence', 'logistic']
        for baseline in ['static_5', 'training_prevalence']
    ])
    df['contract_year'] = pd.to_datetime(df.contract_end_date).dt.year
    cohorts = []
    for dimension in ('contract_year', 'expansion_potential_signal'):
        for cohort, group in df.groupby(dimension, dropna=False):
            cohorts.append({
                'dimension': dimension, 'cohort': str(cohort), 'n': len(group),
                **probability_metrics(group[TARGET].to_numpy(), group[PRIMARY_SCORE].to_numpy()),
                **paired_account_ci(group, PRIMARY_SCORE, 'static_5', repetitions=repetitions),
            })
    cohorts = pd.DataFrame(cohorts)
    df['risk_bucket'] = df[PRIMARY_SCORE].map(assign_risk_bucket)
    buckets = df.groupby('risk_bucket').agg(
        arrays=(TARGET, 'size'), churns=(TARGET, 'sum'), observed_rate=(TARGET, 'mean'),
        signed_historical_profit_usd=('historical_15yr_net_profit_usd', 'sum'),
    ).reindex(['CRITICAL', 'HIGH', 'MEDIUM', 'LOW'])
    buckets[['arrays', 'churns', 'signed_historical_profit_usd']] = buckets[
        ['arrays', 'churns', 'signed_historical_profit_usd']].fillna(0)
    if not np.isclose(buckets.signed_historical_profit_usd.sum(), df.historical_15yr_net_profit_usd.sum()):
        raise AssertionError('Historical exposure reconciliation failed')
    primary = metrics.set_index('model').loc[PRIMARY_SCORE]
    vs_prev = ci.loc[(ci.model == PRIMARY_SCORE) & (ci.baseline == 'training_prevalence')].iloc[0]
    conclusion = ('The paired CI supports a Brier gain over prevalence on these fixed predictions; '
                  'independent deployment validation is still needed.'
                  if vs_prev.gain_ci_low > 0 else
                  'No reliable Brier benefit over a fold-trained portfolio rate is established. '
                  'Retention targeting and a production champion claim are unsupported.')
    lines = [
        '# Grouped out-of-fold churn evaluation',
        '',
        f"Run: {manifest['run_id']} | Model: {manifest['model_version']}",
        f"Prediction source: grouped OOF | Labeled arrays: {len(df)} | Accounts: {df.account_id.nunique()}",
        f"Observed 12-month array-event rate: {y.mean():.4%}",
        '',
        'Outer folds keep accounts intact. Capacity bins fit within each training fold. '
        'Calibrators fit on inner account-grouped OOF scores; outer labels never fit the mapping.',
        'This is cross-sectional validation, not historical back-testing. Observation dates and label maturity are unverified.',
        '',
        '## Probability metrics',
        '',
        'Average precision is reported as AP, not trapezoidal PR area. Lower Brier/log loss is better. '
        'Positive bias means over-forecasting churn. Candidates and fixed weights were specified before this run; no outer-fold threshold tuning.',
        '',
        metrics.to_markdown(index=False, floatfmt='.6f'),
        '',
        f'Prespecified candidate: {PRIMARY_SCORE}.',
        improvement_text(primary.brier_lift_vs_5_pct),
        conclusion,
        '',
        '## Paired account bootstrap intervals',
        '',
        f'{repetitions} resamples; seed 20261011. Positive gain = lower Brier. '
        'Intervals condition on predictions and exclude refitting/model-selection uncertainty.',
        '',
        ci.to_markdown(index=False, floatfmt='.6f'),
        '',
        '## Diagnostic cohorts',
        '',
        'Contract expiry year and expansion tags are proxy segments, not product families or historical windows. '
        'Cohort variability does not prove feature usefulness.',
        '',
        cohorts.to_markdown(index=False, floatfmt='.6f'),
        '',
        '## Signed historical exposure by OOF risk bucket',
        '',
        'Policy thresholds: CRITICAL >=0.70; HIGH >=0.40; MEDIUM >=0.15; LOW <0.15. '
        'Buckets do not establish retention uplift. Empty-bucket observed rates are undefined.',
        '',
        buckets.to_markdown(floatfmt='.6f'),
        '',
        f'Signed labeled historical profit total: ${df.historical_15yr_net_profit_usd.sum():,.2f}. '
        'This is neither expected future loss nor discounted CLV.',
        '',
        '## Limitations and recommendation',
        '',
        'Array labels do not establish account cancellation. Future expansion outcomes, product families, '
        'observation timestamps, event durations and future cash flows are absent. '
        'A 15-year horizon requires explicit scenarios and cannot be claimed as observed renewal performance.',
        conclusion,
        'Live Snowflake procedure compilation/execution remains unverified.',
        '',
    ]
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)
    for name, table in [('metrics.csv', metrics), ('bootstrap_intervals.csv', ci), ('cohorts.csv', cohorts)]:
        write_once(root / name, table.to_csv(index=False))
    write_once(root / 'historical_exposure_buckets.csv', buckets.to_csv())
    report = '\n'.join(lines)
    write_once(root / 'backtesting_report.md', report)
    return {'metrics': metrics, 'intervals': ci, 'cohorts': cohorts, 'buckets': buckets, 'report': report}


if __name__ == '__main__':
    from src.pipeline import main
    main(['evaluate'])

'''
write_embedded('src/backtesting.py', MODULE_CODE, [243], '8a8feeced9d29c795df585082a03d2a8ed54d91088a93ec2a975262e1f260223')

In [ ]:
# Embedded src/config.py
MODULE_CODE = r'''"""Canonical repository paths and exported scoring contract."""
from pathlib import Path
ROOT = Path(__file__).resolve().parents[1]
DATA_PATH = ROOT / 'data' / 'Case_Study_2_CLV_Predictive_Analytics.csv'
OUTPUTS = ROOT / 'outputs'
MODEL_VERSION = 'v2.1.0-grouped-sigmoid-unweighted'
REFERENCE_DATE = '2026-10-01'
PRIMARY_SCORE = 'unweighted_ensemble_sigmoid'
SCORE_COLUMNS = [
    'account_id', 'array_id', 'contract_end_date', 'historical_15yr_net_profit_usd',
    'current_capacity_tb', 'utilization_rate_pct', 'raw_utilization_rate_pct',
    'utilization_capped', 'is_over_utilized_raw', 'is_negative_profit',
    'expansion_potential_signal', 'expansion_signal_encoded',
    'static_baseline_churn_rate', 'xgb_churn_probability', 'rf_churn_probability',
    'ensemble_churn_probability', 'churn_risk_bucket', 'shap_top_feature_1',
    'shap_top_feature_1_value', 'shap_top_feature_2', 'shap_top_feature_2_value',
    'shap_top_feature_3', 'shap_top_feature_3_value', 'scored_at', 'model_version',
]

'''
write_embedded('src/config.py', MODULE_CODE, [18], 'dd6eaec1bc5dcbdec24e9bc45407b68b344afc0163a3f6e384fa6027d3004e27')

In [ ]:
# Embedded src/contracts.py
MODULE_CODE = r'''"""Validate the 25-column CSV contract before any scoring export is written."""
import numpy as np
import pandas as pd
from src.config import SCORE_COLUMNS
from src.data_cleaning import EXPANSION_SIGNAL_MAP
from src.utils import assign_risk_bucket


def validate_scored_output(df):
    if list(df.columns) != SCORE_COLUMNS or df.empty or df.isna().any().any():
        raise ValueError('Scoring output must have the exact populated 25-column contract')
    if df.duplicated(['account_id', 'array_id', 'model_version']).any():
        raise ValueError('Duplicate scoring keys')
    for c in ('account_id', 'array_id', 'model_version'):
        if df[c].astype(str).str.len().gt(50).any() or df[c].astype(str).str.strip().eq('').any():
            raise ValueError(f'{c} does not fit VARCHAR(50)')
    limits = {
        'historical_15yr_net_profit_usd': 10**13,
        'current_capacity_tb': 10**8,
        'utilization_rate_pct': 10**4,
        'raw_utilization_rate_pct': 10**4,
    }
    for c, bound in limits.items():
        if not np.isfinite(df[c]).all() or df[c].abs().ge(bound).any():
            raise ValueError(f'{c} exceeds warehouse precision')
    if (df.current_capacity_tb <= 0).any() or (df.raw_utilization_rate_pct < 0).any():
        raise ValueError('Exported capacity/utilization is invalid after rounding')
    if not np.allclose(df.utilization_rate_pct, df.raw_utilization_rate_pct, atol=1e-6, rtol=0):
        raise ValueError('Raw utilization columns disagree')
    if not np.allclose(df.utilization_capped, df.raw_utilization_rate_pct.clip(upper=1), atol=1e-6, rtol=0):
        raise ValueError('Capped utilization disagrees with raw utilization')
    if not df.is_over_utilized_raw.eq(df.raw_utilization_rate_pct > 1).all():
        raise ValueError('Over-utilization flag lost its meaning after export rounding')
    if not df.is_negative_profit.eq(df.historical_15yr_net_profit_usd < 0).all():
        raise ValueError('Negative-profit flag lost its meaning after export rounding')
    expected_signal = df.expansion_potential_signal.map(EXPANSION_SIGNAL_MAP)
    if expected_signal.isna().any() or not df.expansion_signal_encoded.eq(expected_signal).all():
        raise ValueError('Expansion signal/export code mismatch')
    if not np.allclose(df.static_baseline_churn_rate, .05, atol=1e-6, rtol=0):
        raise ValueError('Static baseline must be 5%')
    for c in ('xgb_churn_probability', 'rf_churn_probability', 'ensemble_churn_probability'):
        p = df[c].to_numpy()
        if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError('Exported probabilities must be finite in [0,1]')
    if not df.churn_risk_bucket.eq(df.ensemble_churn_probability.map(assign_risk_bucket)).all():
        raise ValueError('Exported buckets disagree with the probability policy')
    for i in range(1, 4):
        if not np.isfinite(df[f'shap_top_feature_{i}_value']).all():
            raise ValueError('Nonfinite SHAP values')
        if df[f'shap_top_feature_{i}'].str.len().gt(100).any():
            raise ValueError('SHAP feature name exceeds warehouse width')
    if pd.to_datetime(df.contract_end_date, errors='raise').isna().any():
        raise ValueError('Missing contract date')
    if pd.to_datetime(df.scored_at, errors='raise').isna().any():
        raise ValueError('Missing scoring timestamp')
    return True

'''
write_embedded('src/contracts.py', MODULE_CODE, [56], 'b304ff224224d7e4494000e8685b1d8609d32c69df409aec74634e9a7c5b02a2')

In [ ]:
# Embedded src/data_cleaning.py
MODULE_CODE = r'''"""Validate snapshots; preserve anomalies without learning batch statistics."""
from pathlib import Path
import numpy as np
import pandas as pd
TARGET = 'churn_event_next_12m'
SIGNALS = ('UNKNOWN', 'Low', 'Medium', 'High')
EXPANSION_SIGNAL_MAP = dict(zip(SIGNALS, range(4)))  # Export code, not an ML ordering.
REQUIRED = ('account_id', 'array_id', 'contract_end_date',
            'historical_15yr_net_profit_usd', 'current_capacity_tb',
            'utilization_rate_pct', 'expansion_potential_signal')


def validate_raw_frame(data, require_target=False):
    if not isinstance(data, (str, Path, pd.DataFrame)):
        raise ValueError('Provide a raw DataFrame or CSV path, not precomputed feature arrays')
    df = pd.read_csv(data, dtype={'account_id': str, 'array_id': str}) if isinstance(data, (str, Path)) else data.copy()
    missing = set(REQUIRED) - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if df.empty:
        raise ValueError('Snapshot must contain at least one row')
    for col in ('account_id', 'array_id'):
        if df[col].isna().any() or df[col].astype(str).str.strip().eq('').any():
            raise ValueError(f'{col} must be populated')
        df[col] = df[col].astype(str)
    if df.duplicated(['account_id', 'array_id']).any():
        raise ValueError('Duplicate account/array keys')
    for col in ('historical_15yr_net_profit_usd', 'current_capacity_tb', 'utilization_rate_pct'):
        df[col] = pd.to_numeric(df[col], errors='raise')
        if not np.isfinite(df[col]).all():
            raise ValueError(f'{col} must be finite and populated')
    if (df.current_capacity_tb <= 0).any():
        raise ValueError('Capacity must be positive')
    if (df.utilization_rate_pct < 0).any():
        raise ValueError('Utilization must be nonnegative (fraction units)')
    df['contract_end_date'] = pd.to_datetime(df.contract_end_date, errors='raise')
    if df.contract_end_date.isna().any():
        raise ValueError('Contract dates must be populated')
    df['expansion_potential_signal'] = df.expansion_potential_signal.fillna('UNKNOWN')
    if not df.expansion_potential_signal.isin(SIGNALS).all():
        raise ValueError('Unsupported expansion category; supported: UNKNOWN/Low/Medium/High')
    if TARGET not in df:
        if require_target:
            raise ValueError('Training requires the churn target column')
        df[TARGET] = np.nan
    df[TARGET] = pd.to_numeric(df[TARGET], errors='raise')
    if not df[TARGET].dropna().isin([0, 1]).all():
        raise ValueError('Churn target must be 0, 1, or missing')
    return df.reset_index(drop=True)


def audit_anomalies(df):
    audit = {'total_rows': len(df)}
    for name, mask in {
        'over_utilized': df.utilization_rate_pct > 1,
        'negative_profit': df.historical_15yr_net_profit_usd < 0,
        'missing_churn': df[TARGET].isna(),
        'unknown_expansion': df.expansion_potential_signal.eq('UNKNOWN'),
    }.items():
        audit[f'{name}_count'] = int(mask.sum())
        audit[f'{name}_pct'] = float(mask.mean() * 100)
    audit.update(max_raw_utilization=float(df.utilization_rate_pct.max()),
                 min_profit_usd=float(df.historical_15yr_net_profit_usd.min()))
    return audit


def clean_dataset(filepath_or_df, reference_min_profit=None):
    """Signed-log cleaning uses no fitted offset; targets are never imputed."""
    if reference_min_profit is not None:
        raise ValueError('v2 uses signed log; shifted-profit offsets are no longer supported')
    df = validate_raw_frame(filepath_or_df)
    audit = audit_anomalies(df)
    df['raw_utilization_rate_pct'] = df.utilization_rate_pct
    df['utilization_capped'] = df.utilization_rate_pct.clip(upper=1)
    df['is_over_utilized_raw'] = (df.utilization_rate_pct > 1).astype(int)
    df['is_negative_profit'] = (df.historical_15yr_net_profit_usd < 0).astype(int)
    df['signed_log_profit'] = np.sign(df.historical_15yr_net_profit_usd) * np.log1p(df.historical_15yr_net_profit_usd.abs())
    df['expansion_signal_encoded'] = df.expansion_potential_signal.map(EXPANSION_SIGNAL_MAP)
    labeled = df.loc[df[TARGET].notna()].reset_index(drop=True)
    unlabeled = df.loc[df[TARGET].isna()].reset_index(drop=True)
    labeled[TARGET] = labeled[TARGET].astype(int)
    audit.update(labeled_rows=len(labeled), unlabeled_rows=len(unlabeled),
                 labeled_churn_count=int(labeled[TARGET].sum()),
                 labeled_churn_rate_pct=float(labeled[TARGET].mean() * 100))
    return labeled, unlabeled, audit


if __name__ == '__main__':
    from src.config import DATA_PATH
    print(clean_dataset(DATA_PATH)[2])

'''
write_embedded('src/data_cleaning.py', MODULE_CODE, [90], '535b7bfc8ba3750a3fee5981f53afb6b4a5240ba83b3547f282d64cea8119119')

In [ ]:
# Embedded src/explainability.py
MODULE_CODE = r'''"""Explain the XGB component in raw-margin/log-odds units."""
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb


def xgb_contributions(bundle, raw):
    X = bundle['preprocessor'].transform(raw)
    booster = bundle['xgb'].get_booster()
    values = booster.predict(xgb.DMatrix(X), pred_contribs=True)
    margin = booster.predict(xgb.DMatrix(X), output_margin=True)
    if not np.allclose(values.sum(axis=1), margin, atol=1e-5):
        raise AssertionError('SHAP contributions fail raw-margin additivity')
    prob = 1 / (1 + np.exp(-margin))
    if not np.allclose(prob, bundle['xgb'].predict_proba(X)[:, 1], atol=1e-6):
        raise AssertionError('XGB raw margin does not reconstruct raw probability')
    return values[:, :-1], values[:, -1], X


def extract_top_shap_features(shap_values, feature_names, top_n=3):
    records = []
    for values in shap_values:
        indices = np.argsort(np.abs(values))[::-1][:top_n]
        records.append({item: value for rank, index in enumerate(indices, 1)
                        for item, value in (
                            (f'shap_top_feature_{rank}', feature_names[index]),
                            (f'shap_top_feature_{rank}_value', float(values[index])),
                        )})
    return pd.DataFrame(records)


def generate_shap_analysis(bundle, raw, output_dir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import shap
    values, base, X = xgb_contributions(bundle, raw)
    names = bundle['preprocessor'].feature_names_
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for kind, filename in [('bar', 'shap_summary_bar.png'), ('dot', 'shap_beeswarm.png')]:
        shap.summary_plot(values, X, feature_names=names, plot_type=kind, show=False)
        plt.title('XGB component contributions in raw margin (log-odds)')
        plt.tight_layout()
        plt.savefig(output_dir / filename, dpi=150, bbox_inches='tight')
        plt.close('all')
    for name, filename in [('utilization_capped', 'shap_dependence_utilization.png'),
                           ('signed_log_profit', 'shap_dependence_profit.png')]:
        shap.dependence_plot(names.index(name), values, X, feature_names=names,
                             interaction_index=None, show=False)
        plt.ylabel('XGB contribution (log-odds)')
        plt.tight_layout()
        plt.savefig(output_dir / filename, dpi=150, bbox_inches='tight')
        plt.close('all')
    return values, base

'''
write_embedded('src/explainability.py', MODULE_CODE, [56], 'd9d03787320895b654b77a43a101d5ece43dfa3863dd51d1cafd4e5ae44f7ef8')

In [ ]:
# Embedded src/feature_engineering.py
MODULE_CODE = r'''"""Fitted, serializable preprocessing with explicit feature order."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from src.config import REFERENCE_DATE
from src.data_cleaning import validate_raw_frame, SIGNALS


class CLVPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, reference_date=REFERENCE_DATE):
        self.reference_date = reference_date

    def fit(self, X, y=None):
        df = validate_raw_frame(X)
        cuts = np.unique(np.quantile(df.current_capacity_tb, [.25, .5, .75]))
        self.capacity_edges_ = np.r_[-np.inf, cuts, np.inf]
        self.feature_names_ = [
            'current_capacity_tb', 'is_over_utilized_raw', 'utilization_capped',
            'is_negative_profit', 'signed_log_profit', 'days_to_contract_end',
            'contract_year', 'contract_quarter', 'profit_per_tb', 'stress_index',
            'capacity_bin', *[f'expansion_{s.lower()}' for s in SIGNALS],
        ]
        self.fit_rows_ = len(df)
        return self

    def transform(self, X):
        check_is_fitted(self, 'capacity_edges_')
        df = validate_raw_frame(X)
        profit = df.historical_15yr_net_profit_usd.to_numpy()
        capped = df.utilization_rate_pct.clip(upper=1)
        neg = (profit < 0).astype(int)
        f = pd.DataFrame({
            'current_capacity_tb': df.current_capacity_tb,
            'is_over_utilized_raw': (df.utilization_rate_pct > 1).astype(int),
            'utilization_capped': capped, 'is_negative_profit': neg,
            'signed_log_profit': np.sign(profit) * np.log1p(np.abs(profit)),
            'days_to_contract_end': (df.contract_end_date - pd.Timestamp(self.reference_date)).dt.days.clip(lower=0),
            'contract_year': df.contract_end_date.dt.year,
            'contract_quarter': df.contract_end_date.dt.quarter,
            'profit_per_tb': profit / df.current_capacity_tb,
            'stress_index': capped * (1 + neg),
            'capacity_bin': np.searchsorted(self.capacity_edges_[1:-1], df.current_capacity_tb, side='left') + 1,
        })
        for s in SIGNALS:
            f[f'expansion_{s.lower()}'] = df.expansion_potential_signal.eq(s).astype(int)
        result = f[self.feature_names_].to_numpy(dtype=float)
        if not np.isfinite(result).all():
            raise ValueError('Transformed features must be finite')
        return result

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'feature_names_')
        return np.asarray(self.feature_names_, dtype=object)


def build_feature_pipeline(df_labeled, df_unlabeled):
    """Exploration helper only; model CV fits its own fold-local transformer."""
    prep = CLVPreprocessor().fit(df_labeled)
    unseen = prep.transform(df_unlabeled) if len(df_unlabeled) else np.empty((0, len(prep.feature_names_)))
    return prep.transform(df_labeled), df_labeled['churn_event_next_12m'].to_numpy(), unseen, prep.feature_names_

'''
write_embedded('src/feature_engineering.py', MODULE_CODE, [61], '935be135b48f8791f6ec995978cd31290bb8964552c64065b0639931d55567e9')

In [ ]:
# Embedded src/model_training.py
MODULE_CODE = r'''"""Account-grouped evaluation, inner-OOF calibration, and reusable inference bundles."""
import hashlib
import importlib.metadata
import json
import pickle
import platform
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from src.config import OUTPUTS, MODEL_VERSION, PRIMARY_SCORE, REFERENCE_DATE
from src.data_cleaning import TARGET, validate_raw_frame
from src.feature_engineering import CLVPreprocessor

SEED = 42
DEFAULT_PARAMS = {'xgb_estimators': 150, 'rf_estimators': 200}


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def frame_hash(df):
    """Hash raw labeled inputs in canonical column order, including row order."""
    cols = ['account_id', 'array_id', 'contract_end_date',
            'historical_15yr_net_profit_usd', 'current_capacity_tb',
            'utilization_rate_pct', 'expansion_potential_signal', TARGET]
    return hashlib.sha256(df[cols].to_csv(index=False, date_format='%Y-%m-%d').encode()).hexdigest()


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


class ProbabilityMap:
    """Sigmoid or isotonic mapping fitted exclusively to inner OOF probabilities."""
    def __init__(self, method='sigmoid'):
        self.method = method

    def fit(self, p, y):
        if np.unique(y).size != 2:
            raise ValueError('Calibration requires both target classes')
        if self.method == 'sigmoid':
            self.estimator_ = LogisticRegression(C=1e6, max_iter=1000, random_state=SEED)
            self.estimator_.fit(logit(p), y)
        elif self.method == 'isotonic':
            self.estimator_ = IsotonicRegression(out_of_bounds='clip').fit(p, y)
        else:
            raise ValueError('Mapping must be sigmoid or isotonic')
        return self

    def predict(self, p):
        if self.method == 'sigmoid':
            return self.estimator_.predict_proba(logit(p))[:, 1]
        return self.estimator_.predict(p)


def grouped_splits(df, n_splits):
    groups = df.account_id.to_numpy()
    if df.account_id.nunique() < n_splits:
        raise ValueError('Insufficient distinct accounts for grouped folds')
    for train, val in GroupKFold(n_splits=n_splits).split(df, df[TARGET], groups):
        if set(groups[train]) & set(groups[val]):
            raise AssertionError('Account overlap between training and validation')
        if df.iloc[train][TARGET].nunique() != 2:
            raise ValueError('Each training fold requires both target classes')
        yield train, val


def fit_pair(train, weighted, seed, params=None, reference_date=REFERENCE_DATE):
    params = {**DEFAULT_PARAMS, **(params or {})}
    prep = CLVPreprocessor(reference_date=reference_date).fit(train)
    X = prep.transform(train)
    y = train[TARGET].to_numpy(dtype=int)
    positive_weight = (y == 0).sum() / (y == 1).sum() if weighted else 1.0
    xgb = XGBClassifier(
        n_estimators=params['xgb_estimators'], max_depth=4, learning_rate=.05,
        subsample=.8, colsample_bytree=.8, reg_alpha=.1, reg_lambda=1.,
        scale_pos_weight=positive_weight, eval_metric='logloss',
        random_state=seed, n_jobs=2,
    ).fit(X, y)
    rf = RandomForestClassifier(
        n_estimators=params['rf_estimators'], max_depth=8,
        min_samples_split=10, min_samples_leaf=4, max_features='sqrt',
        class_weight='balanced' if weighted else None,
        random_state=seed, n_jobs=2,
    ).fit(X, y)
    return {'preprocessor': prep, 'xgb': xgb, 'rf': rf}


def predict_pair(pair, data):
    X = pair['preprocessor'].transform(data)
    a = pair['xgb'].predict_proba(X)[:, 1]
    b = pair['rf'].predict_proba(X)[:, 1]
    return {'xgb': a, 'rf': b, 'ensemble': .6 * a + .4 * b}


def inner_predictions(train, weighted, n_splits, seed, params=None, reference_date=REFERENCE_DATE):
    inner = {name: np.full(len(train), np.nan) for name in ('xgb', 'rf', 'ensemble')}
    for fold, (tr, va) in enumerate(grouped_splits(train, n_splits), 1):
        pair = fit_pair(train.iloc[tr], weighted, seed + fold, params, reference_date=reference_date)
        pred = predict_pair(pair, train.iloc[va])
        for name in inner:
            inner[name][va] = pred[name]
    if not all(np.isfinite(p).all() for p in inner.values()):
        raise AssertionError('Incomplete inner out-of-fold predictions')
    return inner


def fit_maps(predictions, y):
    return {f'{name}_{method}': ProbabilityMap(method).fit(p, y)
            for name, p in predictions.items() for method in ('sigmoid', 'isotonic')}


def predict_bundle(bundle, raw):
    pred = predict_pair(bundle, raw)
    return {
        'xgb': bundle['calibrators']['xgb_sigmoid'].predict(pred['xgb']),
        'rf': bundle['calibrators']['rf_sigmoid'].predict(pred['rf']),
        'ensemble': bundle['calibrators']['ensemble_sigmoid'].predict(pred['ensemble']),
        'prevalence': np.full(len(raw), bundle['training_prevalence']),
    }


def train_and_evaluate_models(raw, n_splits=5, inner_splits=3, output_root=OUTPUTS, params=None):
    """Evaluate prespecified candidates; no winner or threshold selected on outer labels."""
    raw = validate_raw_frame(raw, require_target=True)
    labeled = raw.loc[raw[TARGET].notna()].reset_index(drop=True)
    if labeled.empty:
        raise ValueError('Training needs labeled rows')
    labeled[TARGET] = labeled[TARGET].astype(int)
    run_id = uuid.uuid4().hex
    run_dir = Path(output_root) / 'runs' / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    y = labeled[TARGET].to_numpy()
    columns = ['account_id', 'array_id', 'contract_end_date',
               'historical_15yr_net_profit_usd', 'expansion_potential_signal', TARGET]
    evidence = labeled[columns].copy()
    evidence['prediction_source'] = 'grouped_oof'
    evidence['run_id'] = run_id
    evidence['fold_id'] = -1
    evidence['static_5'] = .05
    evidence['training_prevalence'] = np.nan
    evidence['logistic'] = np.nan
    for weight in ('weighted', 'unweighted'):
        for name in ('xgb', 'rf', 'ensemble'):
            for mapping in ('raw', 'sigmoid', 'isotonic'):
                evidence[f'{weight}_{name}_{mapping}'] = np.nan
    folds = []
    for fold, (tr, va) in enumerate(grouped_splits(labeled, n_splits), 1):
        train, val = labeled.iloc[tr], labeled.iloc[va]
        print(f'Outer account fold {fold}/{n_splits}: train={len(tr)}, validation={len(va)}', flush=True)
        evidence.loc[va, 'fold_id'] = fold
        evidence.loc[va, 'training_prevalence'] = float(train[TARGET].mean())
        prep = CLVPreprocessor().fit(train)
        logistic = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=2000, random_state=SEED))
        logistic.fit(prep.transform(train), train[TARGET])
        evidence.loc[va, 'logistic'] = logistic.predict_proba(prep.transform(val))[:, 1]
        folds.append({
            'fold_id': fold, 'train_accounts': sorted(train.account_id.unique().tolist()),
            'validation_accounts': sorted(val.account_id.unique().tolist()),
            'train_rows': len(tr), 'validation_rows': len(va),
            'capacity_edges': [None if not np.isfinite(x) else float(x) for x in prep.capacity_edges_],
            'training_prevalence': float(train[TARGET].mean()),
        })
        for weighted in (True, False):
            prefix = 'weighted' if weighted else 'unweighted'
            inner = inner_predictions(train.reset_index(drop=True), weighted, inner_splits, SEED + 100 * fold, params)
            maps = fit_maps(inner, train[TARGET].to_numpy())
            pair = fit_pair(train, weighted, SEED + fold, params)
            pred = predict_pair(pair, val)
            for name, p in pred.items():
                evidence.loc[va, f'{prefix}_{name}_raw'] = p
                for method in ('sigmoid', 'isotonic'):
                    evidence.loc[va, f'{prefix}_{name}_{method}'] = maps[f'{name}_{method}'].predict(p)
    # Final inference mapping uses grouped inner OOF on all labeled data.
    print('Fitting final unweighted inference bundle and grouped calibrators', flush=True)
    final_inner = inner_predictions(labeled, False, inner_splits, SEED + 1000, params)
    bundle = fit_pair(labeled, False, SEED, params)
    bundle.update(
        calibrators=fit_maps(final_inner, y), training_prevalence=float(y.mean()),
        model_version=MODEL_VERSION, training_run_id=run_id,
        primary_score=PRIMARY_SCORE, target=TARGET,
        reference_date=bundle['preprocessor'].reference_date,
        shap_units='xgb_raw_margin_log_odds',
    )
    bundle_path = run_dir / 'inference_bundle.pkl'
    with bundle_path.open('wb') as f:
        pickle.dump(bundle, f)
    evidence_path = run_dir / 'evaluation_predictions.csv'
    evidence.to_csv(evidence_path, index=False, date_format='%Y-%m-%d')
    snapshot_path = run_dir / 'training_snapshot.csv'
    raw.to_csv(snapshot_path, index=False, date_format='%Y-%m-%d')
    versions = {name: importlib.metadata.version(name) for name in
                ('numpy', 'pandas', 'scikit-learn', 'xgboost', 'shap', 'matplotlib', 'tabulate')}
    manifest = {
        'run_id': run_id, 'model_version': MODEL_VERSION,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'python_version': platform.python_version(), 'platform': platform.platform(),
        'validation_design': 'account_grouped_outer_cv_inner_oof_calibration',
        'prediction_source': 'grouped_oof', 'outer_folds': n_splits,
        'inner_folds': inner_splits, 'seed': SEED, 'parameters': {**DEFAULT_PARAMS, **(params or {})},
        'labeled_rows': len(labeled), 'unlabeled_rows': len(raw) - len(labeled),
        'labeled_input_sha256': frame_hash(labeled),
        'evaluation_sha256': sha256_file(evidence_path), 'bundle_sha256': sha256_file(bundle_path),
        'training_snapshot_sha256': sha256_file(snapshot_path),
        'primary_score': PRIMARY_SCORE,
        'candidate_policy': 'Prespecified unweighted sigmoid ensemble; not a selected champion',
        'feature_names': bundle['preprocessor'].feature_names_,
        'versions': versions, 'folds': folds,
        'source_sha256': {p.name: sha256_file(p) for p in Path(__file__).parent.glob('*.py')},
        'warehouse_source_sha256': {p.name: sha256_file(p) for p in (Path(__file__).parent.parent / 'sql').glob('*.sql')},
        'limitations': ['No observation timestamps or verified label maturity',
                        'No untouched final test or demonstrated account cancellation labels',
                        'No product family, expansion outcomes, or future cash-flow labels',
                        'Live Snowflake execution unverified'],
    }
    (run_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    # Only update latest pointer after the complete artifact set has been saved.
    (Path(output_root) / 'latest_run.json').write_text(
        json.dumps({'run_id': run_id}, indent=2), encoding='utf-8')
    return {'run_dir': run_dir, 'manifest': manifest, 'evaluation': evidence, 'bundle': bundle}


def latest_run(output_root=OUTPUTS):
    root = Path(output_root)
    pointer = json.loads((root / 'latest_run.json').read_text(encoding='utf-8'))
    return root / 'runs' / pointer['run_id']


def load_bundle(run_dir=None):
    run_dir = Path(run_dir) if run_dir is not None else latest_run()
    manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
    path = run_dir / 'inference_bundle.pkl'
    if sha256_file(path) != manifest['bundle_sha256']:
        raise ValueError('Inference bundle hash does not match manifest')
    with path.open('rb') as f:
        bundle = pickle.load(f)
    if bundle['training_run_id'] != manifest['run_id'] or bundle['model_version'] != manifest['model_version']:
        raise ValueError('Bundle lineage mismatch')
    return bundle, manifest


if __name__ == '__main__':
    from src.pipeline import main
    main(['train'])

'''
write_embedded('src/model_training.py', MODULE_CODE, [257], 'f8946fc64d4b4a59bb4302e9364fdb9f6fcf716d1957afe61a46df3b3e0d7336')

In [ ]:
# Embedded src/pipeline.py
MODULE_CODE = r'''"""Reproducible module entry point: python -m src.pipeline run|train|evaluate|score."""
import argparse
import json
import shutil
from pathlib import Path
import pandas as pd
from src.config import ROOT, DATA_PATH, OUTPUTS
from src.data_cleaning import validate_raw_frame, TARGET
from src.model_training import train_and_evaluate_models, latest_run, load_bundle
from src.backtesting import read_evidence, run_backtesting_suite
from src.scoring_pipeline import generate_production_scores
from src.explainability import generate_shap_analysis
from src.submission_analytics import generate_submission_analytics


def evaluate(run_dir=None):
    run_dir = Path(run_dir) if run_dir else latest_run()
    evidence, manifest = read_evidence(run_dir)
    result = run_backtesting_suite(evidence, manifest, run_dir)
    shutil.copyfile(run_dir / 'backtesting_report.md', OUTPUTS / 'backtesting_report.md')
    return result


def score(data=DATA_PATH, run_dir=None, output=None, cash_flows=None, product_mapping=None):
    df, metadata, path = generate_production_scores(data, run_dir=run_dir, output_path=output)
    generate_submission_analytics(path.parent, run_dir, cash_flows, product_mapping)
    # Convenience export is explicitly production scoring, never used for validation.
    if output is None:
        shutil.copyfile(path, OUTPUTS / 'CLV_SCORED_OUTPUT.csv')
        (OUTPUTS / 'latest_score_run.json').write_text(json.dumps({
            'score_run_id': metadata['score_run_id'], 'directory': path.parent.relative_to(OUTPUTS).as_posix(),
        }, indent=2), encoding='utf-8')
    print(f'Scored {len(df)} arrays: {path}', flush=True)
    return df, metadata, path


def refresh_readme_results(run_dir, result):
    readme = ROOT / 'README.md'
    if not readme.exists():
        return
    text = readme.read_text(encoding='utf-8-sig')
    start, end = '<!-- RESULTS_START -->', '<!-- RESULTS_END -->'
    if start not in text or end not in text:
        return
    metrics = result['metrics'].set_index('model')
    names = ['static_5', 'training_prevalence', 'logistic', 'weighted_ensemble_raw',
             'unweighted_xgb_raw', 'unweighted_xgb_sigmoid', 'unweighted_rf_raw', 'unweighted_rf_sigmoid',
             'unweighted_ensemble_raw', 'unweighted_ensemble_sigmoid']
    table = metrics.loc[names, ['mean_probability', 'bias', 'brier', 'roc_auc', 'average_precision', 'brier_lift_vs_5_pct']]
    content = '\n\nRun: ' + Path(run_dir).name + '\n\n' + table.to_markdown(floatfmt='.6f') + '\n\n'
    text = text.split(start)[0] + start + content + end + text.split(end, 1)[1]
    readme.write_text(text, encoding='utf-8')


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=['run', 'train', 'evaluate', 'score'])
    parser.add_argument('--data', type=Path, default=DATA_PATH)
    parser.add_argument('--run-dir', type=Path)
    parser.add_argument('--output', type=Path, help='Fresh score CSV path (score command)')
    parser.add_argument('--cash-flows', type=Path, help='Explicit keyed years 1-15 future cash flows; otherwise unit sensitivity only')
    parser.add_argument('--product-mapping', type=Path, help='Actual account/array/product-family lookup')
    args = parser.parse_args(argv)
    if args.command in ('run', 'train'):
        result = train_and_evaluate_models(args.data)
        print(f"Training run saved: {result['run_dir']}", flush=True)
        if args.command == 'train':
            return
        evaluation = evaluate(result['run_dir'])
        bundle = result['bundle']
        raw = validate_raw_frame(args.data)
        generate_shap_analysis(bundle, raw, OUTPUTS / 'shap_plots')
        score(args.data, result['run_dir'], cash_flows=args.cash_flows, product_mapping=args.product_mapping)
        refresh_readme_results(result['run_dir'], evaluation)
    elif args.command == 'evaluate':
        evaluate(args.run_dir)
    else:
        score(args.data, args.run_dir, args.output, args.cash_flows, args.product_mapping)


if __name__ == '__main__':
    main()

'''
write_embedded('src/pipeline.py', MODULE_CODE, [82], '98c0992653ff565e74fa8b810b824ef4fe1be06f9341990f31b8a751251fb44c')

In [ ]:
# Embedded src/scenarios.py
MODULE_CODE = r'''"""Explicit assumption sensitivity, never a fitted survival/CLV forecast."""
import numpy as np


def survival_scenario(annual_churn, years=15):
    p = np.asarray(annual_churn, dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any() or not isinstance(years, int) or years < 1:
        raise ValueError('Scenario requires finite p in [0,1] and positive years')
    return (1 - p[..., None]) ** np.arange(1, years + 1)


def discounted_clv_scenario(annual_cash_flows, annual_churn, discount_rate=.1):
    cash = np.asarray(annual_cash_flows, dtype=float)
    if cash.ndim != 1 or not len(cash) or not np.isfinite(cash).all() or not np.isfinite(discount_rate) or discount_rate <= -1:
        raise ValueError('Provide finite annual cash flows and discount_rate > -1')
    survival = survival_scenario(annual_churn, len(cash))
    return np.sum(survival * cash / (1 + discount_rate) ** np.arange(1, len(cash) + 1), axis=-1)

'''
write_embedded('src/scenarios.py', MODULE_CODE, [17], '307abf37293196f306e8462ad5894f7b7d30608ef45780c6ce6d3bf321d2dba1')

In [ ]:
# Embedded src/scoring_pipeline.py
MODULE_CODE = r'''"""Transform snapshots with saved state; keep production scores out of evaluation."""
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from src.config import DATA_PATH, OUTPUTS, SCORE_COLUMNS
from src.data_cleaning import validate_raw_frame, EXPANSION_SIGNAL_MAP
from src.model_training import load_bundle, predict_bundle, sha256_file
from src.explainability import xgb_contributions, extract_top_shap_features
from src.utils import assign_risk_bucket
from src.contracts import validate_scored_output


def generate_production_scores(csv_path=DATA_PATH, run_dir=None, output_path=None):
    raw = validate_raw_frame(csv_path)
    bundle, training_manifest = load_bundle(run_dir)
    pred = predict_bundle(bundle, raw)
    values, base, _ = xgb_contributions(bundle, raw)
    names = bundle['preprocessor'].feature_names_
    top = extract_top_shap_features(values, names)
    score_run_id = uuid.uuid4().hex
    scored_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    df = pd.DataFrame({
        'account_id': raw.account_id, 'array_id': raw.array_id,
        'contract_end_date': raw.contract_end_date.dt.strftime('%Y-%m-%d'),
        'historical_15yr_net_profit_usd': raw.historical_15yr_net_profit_usd.round(2),
        'current_capacity_tb': raw.current_capacity_tb.round(2),
        'utilization_rate_pct': raw.utilization_rate_pct.round(6),
        'raw_utilization_rate_pct': raw.utilization_rate_pct.round(6),
        'utilization_capped': raw.utilization_rate_pct.clip(upper=1).round(6),
        'is_over_utilized_raw': raw.utilization_rate_pct > 1,
        'is_negative_profit': raw.historical_15yr_net_profit_usd < 0,
        'expansion_potential_signal': raw.expansion_potential_signal,
        'expansion_signal_encoded': raw.expansion_potential_signal.map(EXPANSION_SIGNAL_MAP),
        'static_baseline_churn_rate': .05,
        'xgb_churn_probability': np.round(pred['xgb'], 6),
        'rf_churn_probability': np.round(pred['rf'], 6),
        'ensemble_churn_probability': np.round(pred['ensemble'], 6),
        'churn_risk_bucket': [assign_risk_bucket(p) for p in np.round(pred['ensemble'], 6)],
        'scored_at': scored_at, 'model_version': bundle['model_version'],
    })
    for col in top:
        df[col] = top[col].round(6) if col.endswith('_value') else top[col]
    df = df[SCORE_COLUMNS]
    validate_scored_output(df)
    output_path = Path(output_path) if output_path else OUTPUTS / 'score_runs' / score_run_id / 'CLV_SCORED_OUTPUT.csv'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        raise FileExistsError('Scoring runs are immutable; choose a fresh output path')
    df.to_csv(output_path, index=False)
    explanations = {
        'score_run_id': score_run_id, 'model_version': bundle['model_version'],
        'explained_model': 'unweighted_xgb_raw', 'units': 'raw_margin_log_odds',
        'explanation_scope': 'XGB component only, not calibrated ensemble probability or causal effects',
        'rows': [
            {'account_id': a, 'array_id': b, 'base_value': float(v),
             'contributions': dict(zip(names, map(float, row)))}
            for a, b, v, row in zip(raw.account_id, raw.array_id, base, values)
        ],
    }
    explanation_path = output_path.with_name('xgb_explanations.json')
    explanation_path.write_text(json.dumps(explanations, indent=2), encoding='utf-8')
    raw_path = output_path.with_name('input_snapshot.csv')
    raw.to_csv(raw_path, index=False, date_format='%Y-%m-%d')
    metadata = {
        'score_run_id': score_run_id, 'training_run_id': bundle['training_run_id'],
        'model_version': bundle['model_version'], 'scored_at_utc': scored_at,
        'reference_date': bundle['reference_date'], 'target_horizon': 'next_12_months_array_event',
        'snapshot_sha256': sha256_file(raw_path), 'bundle_sha256': training_manifest['bundle_sha256'],
        'original_input_sha256': sha256_file(csv_path) if isinstance(csv_path, (str, Path)) else None,
        'csv_sha256': sha256_file(output_path), 'explanations_sha256': sha256_file(explanation_path),
        'columns': SCORE_COLUMNS, 'rows': len(df), 'shap_units': explanations['units'],
        'primary_score': bundle['primary_score'],
        'probability_semantics': {
            'xgb': 'sigmoid mapping of raw XGB probability',
            'rf': 'sigmoid mapping of raw RF probability',
            'ensemble': 'sigmoid mapping of 0.6*raw_XGB + 0.4*raw_RF; not the weighted sum of mapped components',
        },
        'policy_thresholds': {'CRITICAL': .70, 'HIGH': .40, 'MEDIUM': .15, 'LOW': 0},
        'deployment_status': 'candidate; live Snowflake execution unverified',
    }
    output_path.with_name('score_manifest.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return df, metadata, output_path


if __name__ == '__main__':
    from src.pipeline import main
    main(['score'])

'''
write_embedded('src/scoring_pipeline.py', MODULE_CODE, [90], 'f45f36a4cbeb953367fdf76842dd51f5e9129801d0e7587d9bbe712f8a40a5c2')

In [ ]:
# Embedded src/submission_analytics.py
MODULE_CODE = r'''"""Account array-event summaries, capacity diagnostics and explicit horizon scenarios."""
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from src.config import PRIMARY_SCORE
from src.data_cleaning import TARGET, validate_raw_frame
from src.model_training import load_bundle, predict_bundle, sha256_file, latest_run
from src.backtesting import read_evidence, probability_metrics, paired_account_ci
from src.scenarios import survival_scenario

KEYS = ['account_id', 'array_id']


def input_fingerprint(value):
    if value is None:
        return None
    if isinstance(value, (str, Path)):
        return sha256_file(value)
    return hashlib.sha256(value.to_csv(index=False).encode('utf-8')).hexdigest()


def account_summary(raw, predictions, probability_column, source):
    """Bounds concern array events conditional on estimated marginals, not cancellation.

    Complete array coverage is required for bounds; missing labels never become zero.
    No independence assumption or account-cancellation target is silently introduced.
    """
    raw = validate_raw_frame(raw)
    p = predictions[KEYS + [probability_column]].copy()
    if p.duplicated(KEYS).any() or p[KEYS].isna().any().any():
        raise ValueError('Prediction keys must be populated and unique')
    values = p[probability_column].to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError('Account aggregation requires finite array probabilities in [0,1]')
    matched = p.merge(raw[KEYS], on=KEYS, how='left', indicator=True, validate='one_to_one')
    if matched['_merge'].ne('both').any():
        raise ValueError('Prediction contains an unknown array key')
    merged = raw.merge(p.rename(columns={probability_column: '_p'}), on=KEYS,
                       how='left', validate='one_to_one')
    records = []
    for account, group in merged.groupby('account_id', sort=True):
        probs = group['_p'].dropna().to_numpy()
        n, covered = len(group), len(probs)
        complete = n == covered
        known_labels = group[TARGET].notna().all()
        records.append({
            'account_id': account, 'arrays_expected': n, 'arrays_scored': covered,
            'complete_probability_coverage': complete,
            'mean_scored_array_event_probability': float(probs.mean()) if covered else np.nan,
            'max_scored_array_event_probability': float(probs.max()) if covered else np.nan,
            'any_array_event_probability_lower_bound': float(probs.max()) if complete else np.nan,
            'any_array_event_probability_upper_bound': float(min(1, probs.sum())) if complete else np.nan,
            'all_arrays_event_probability_lower_bound': float(max(0, probs.sum() - n + 1)) if complete else np.nan,
            'all_arrays_event_probability_upper_bound': float(probs.min()) if complete else np.nan,
            'complete_label_coverage': bool(known_labels),
            'observed_any_array_event_proxy': int(group[TARGET].max()) if known_labels else np.nan,
            'observed_all_arrays_event_proxy': int(group[TARGET].min()) if known_labels else np.nan,
            'signed_historical_profit_usd': float(group.historical_15yr_net_profit_usd.sum()),
            'current_capacity_tb': float(group.current_capacity_tb.sum()),
            'prediction_source': source,
            'semantics': 'array_event_summary_not_account_cancellation',
        })
    return pd.DataFrame(records)


def capacity_signals(raw):
    raw = validate_raw_frame(raw)
    frame = raw[KEYS + ['expansion_potential_signal', 'current_capacity_tb', 'utilization_rate_pct']].copy()
    frame['additional_capacity_to_reach_100pct_tb'] = (
        raw.current_capacity_tb * (raw.utilization_rate_pct - 1).clip(lower=0))
    frame['spare_capacity_at_reported_load_tb'] = (
        raw.current_capacity_tb * (1 - raw.utilization_rate_pct).clip(lower=0))
    frame['reported_over_capacity'] = raw.utilization_rate_pct > 1
    frame['expansion_signal_unknown'] = raw.expansion_potential_signal.eq('UNKNOWN')
    frame['signal_type'] = 'current_load_arithmetic_and_supplied_tag_not_future_prediction'
    return frame


def horizon_scenarios(raw, probabilities, cash_flows=None, discount_rates=(0, .05, .10)):
    """Compare 15-year constant-rate scenarios; optional keyed future cash flows."""
    raw = validate_raw_frame(raw)
    p = np.asarray(probabilities, dtype=float)
    if p.shape != (len(raw),):
        raise ValueError('One probability per array in snapshot order is required')
    survival_scenario(p)  # Validate even when later rate-loop is changed.
    rates = np.asarray(discount_rates, dtype=float)
    if rates.ndim != 1 or not len(rates) or not np.isfinite(rates).all() or (rates <= -1).any():
        raise ValueError('Discount rates must be finite and greater than -1')
    cash = np.ones((len(raw), 15))
    unit = 'one_unit_annual_cash_flow'
    if cash_flows is not None:
        flows = pd.read_csv(cash_flows, dtype=dict.fromkeys(KEYS, str)) if isinstance(cash_flows, (str, Path)) else cash_flows.copy()
        required = set(KEYS + ['year', 'annual_cash_flow_usd'])
        if required - set(flows) or flows.duplicated(KEYS + ['year']).any() or flows.isna().any().any():
            raise ValueError('Cash flows require unique populated account/array/year/cash-flow records')
        values = flows[['year', 'annual_cash_flow_usd']].to_numpy(dtype=float)
        if not np.isfinite(values).all() or not flows.year.isin(range(1, 16)).all():
            raise ValueError('Provide finite cash flows and integer years 1 through 15')
        matrix = flows.pivot(index=KEYS, columns='year', values='annual_cash_flow_usd')
        index = pd.MultiIndex.from_frame(raw[KEYS])
        if set(matrix.index) != set(index) or set(matrix.columns) != set(range(1, 16)):
            raise ValueError('Cash flows must cover exactly the snapshot keys and all 15 years')
        cash = matrix.reindex(index=index, columns=range(1, 16)).to_numpy(dtype=float)
        if not np.isfinite(cash).all():
            raise ValueError('Every array must have all 15 annual cash flows')
        unit = 'usd_from_explicit_user_cash_flows'
    horizons, values = [], []
    for name, annual in [('calibrated_array_event_rate', p), ('static_5pct', np.full(len(raw), .05))]:
        survival = survival_scenario(annual)
        frame = raw[KEYS].copy()
        frame['scenario'] = name
        frame['assumed_annual_event_probability'] = annual
        for year in range(1, 16):
            frame[f'assumed_survival_year_{year}'] = survival[:, year - 1]
        frame['assumed_event_by_year_15'] = 1 - survival[:, -1]
        horizons.append(frame)
        for discount in rates:
            out = raw[KEYS].copy()
            out['scenario'] = name
            out['discount_rate'] = discount
            out['scenario_discounted_value'] = (cash * survival / (1 + discount) ** np.arange(1, 16)).sum(axis=1)
            out['value_units'] = unit
            values.append(out)
    return pd.concat(horizons, ignore_index=True), pd.concat(values, ignore_index=True)


def product_cohort_metrics(evidence, mapping):
    """Actual product-family lookup only; never infer families from other categories."""
    if 'prediction_source' not in evidence or not evidence.prediction_source.eq('grouped_oof').all():
        raise ValueError('Product-cohort evaluation requires grouped OOF evidence')
    mapping = pd.read_csv(mapping, dtype=dict.fromkeys(KEYS, str)) if isinstance(mapping, (str, Path)) else mapping.copy()
    if set(KEYS + ['product_family']) - set(mapping):
        raise ValueError('Product mapping requires account_id, array_id, product_family')
    if mapping.duplicated(KEYS).any() or mapping[KEYS + ['product_family']].isna().any().any():
        raise ValueError('Product mapping must be populated and unique')
    if mapping.product_family.astype(str).str.strip().eq('').any():
        raise ValueError('Product family must be populated')
    joined = evidence.merge(mapping[KEYS + ['product_family']], on=KEYS, how='left', validate='one_to_one')
    if joined.product_family.isna().any():
        raise ValueError('Product mapping does not cover all evaluated arrays')
    rows = []
    for family, group in joined.groupby('product_family'):
        for model in ['unweighted_xgb_sigmoid', 'unweighted_rf_sigmoid', PRIMARY_SCORE]:
            for baseline in ['static_5', 'training_prevalence']:
                rows.append({'product_family': family, 'model': model, 'baseline': baseline,
                             'arrays': len(group), 'accounts': group.account_id.nunique(),
                             **probability_metrics(group[TARGET].to_numpy(), group[model].to_numpy()),
                             **paired_account_ci(group, model, baseline)})
    return pd.DataFrame(rows)


def generate_submission_analytics(score_dir, run_dir=None, cash_flows=None, product_mapping=None):
    """Derived, immutable evidence tied to the actual scoring/training artifacts."""
    score_dir = Path(score_dir)
    score_manifest = json.loads((score_dir / 'score_manifest.json').read_text())
    bundle, manifest = load_bundle(run_dir)
    evidence, _ = read_evidence(run_dir)
    if score_manifest['training_run_id'] != manifest['run_id'] or score_manifest['bundle_sha256'] != manifest['bundle_sha256']:
        raise ValueError('Analytics scoring/training lineage mismatch')
    for file, field in [('input_snapshot.csv', 'snapshot_sha256'), ('CLV_SCORED_OUTPUT.csv', 'csv_sha256')]:
        if sha256_file(score_dir / file) != score_manifest[field]:
            raise ValueError('Analytics input hash mismatch')
    raw = validate_raw_frame(score_dir / 'input_snapshot.csv')
    training_snapshot = (Path(run_dir) if run_dir else latest_run()) / 'training_snapshot.csv'
    if sha256_file(training_snapshot) != manifest['training_snapshot_sha256']:
        raise ValueError('Training snapshot hash mismatch')
    probabilities = predict_bundle(bundle, raw)['ensemble']
    production = raw[KEYS].assign(event_probability=probabilities)
    horizons, values = horizon_scenarios(raw, probabilities, cash_flows)
    tables = {
        'account_array_event_summary.csv': account_summary(raw, production, 'event_probability', 'full_fit_scoring'),
        'account_oof_array_event_summary.csv': account_summary(validate_raw_frame(training_snapshot), evidence, PRIMARY_SCORE, 'grouped_oof'),
        'capacity_expansion_signals.csv': capacity_signals(raw),
        'fifteen_year_scenarios.csv': horizons,
        'discounted_cash_flow_scenarios.csv': values,
    }
    if product_mapping is not None:
        tables['product_family_cohorts.csv'] = product_cohort_metrics(evidence, product_mapping)
    root = score_dir / 'submission_analytics'
    root.mkdir(exist_ok=False)
    for file, table in tables.items():
        table.to_csv(root / file, index=False)
    metadata = {
        'training_run_id': manifest['run_id'], 'score_run_id': score_manifest['score_run_id'],
        'score_manifest_sha256': sha256_file(score_dir / 'score_manifest.json'),
        'evaluation_sha256': manifest['evaluation_sha256'],
        'artifacts': {file: {'rows': len(table), 'sha256': sha256_file(root / file)} for file, table in tables.items()},
        'cash_flow_source_sha256': input_fingerprint(cash_flows),
        'product_mapping_sha256': input_fingerprint(product_mapping),
        'account_semantics': 'Bounds on any/all array events; no account-cancellation prediction or independence assumption',
        'capacity_semantics': 'Current-load arithmetic with uncertain anomaly meaning; no learned future expansion',
        'horizon_semantics': 'Constant annual event-rate assumption; not learned survival or verified renewal',
        'cash_flow_semantics': 'Explicit supplied future USD cash flows, or one unit annually; historical profit is never extrapolated',
        'product_cohort_status': 'evaluated_from_supplied_mapping' if product_mapping is not None else 'unavailable_no_product_mapping',
    }
    (root / 'analytics_manifest.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return root, metadata
'''
write_embedded('src/submission_analytics.py', MODULE_CODE, [], 'aa4f56b2f293b4a9bc696e284d36ade8871b01175e0b80a8ec47a9c676ed03d0')

In [ ]:
# Embedded src/temporal_validation.py
MODULE_CODE = r'''"""Rolling-origin evaluation for real dated array/account/expansion outcomes.

The supplied cross-sectional case-study CSV is deliberately rejected. Test fixtures
exercise implementation, not historical business performance.
"""
import argparse
import json
import pickle
import importlib.metadata
import platform
import uuid
from pathlib import Path
import numpy as np
import pandas as pd
from src.config import OUTPUTS, MODEL_VERSION
from src.data_cleaning import TARGET, REQUIRED, validate_raw_frame
from src.model_training import fit_pair, predict_pair, inner_predictions, fit_maps, sha256_file, SEED, DEFAULT_PARAMS
from src.backtesting import probability_metrics

TARGETS = {TARGET, 'account_cancelled_next_12m', 'capacity_expanded_next_12m'}
DATES = ['observation_date', 'feature_as_of', 'label_window_end', 'label_available_at']


def validate_history(data, target=TARGET, entity_level='array'):
    if target not in TARGETS or entity_level not in ('array', 'account'):
        raise ValueError('Unsupported dated outcome or entity level')
    if target == 'account_cancelled_next_12m' and entity_level != 'account':
        raise ValueError('Account cancellation requires account-level snapshots')
    frame = pd.read_csv(data, dtype={'account_id': str, 'array_id': str}) if isinstance(data, (str, Path)) else data.copy()
    needed = set(REQUIRED) | set(DATES) | {target}
    if entity_level == 'account':
        needed.discard('array_id')
    if target == 'capacity_expanded_next_12m':
        needed.discard('expansion_potential_signal')
    missing = needed - set(frame)
    if missing:
        raise ValueError(f'Dated validation needs explicit snapshot/outcome fields: {sorted(missing)}')
    if entity_level == 'account':
        # Internal preprocessor key only; account IDs are never model features.
        frame['array_id'] = frame['account_id']
    if frame.empty:
        raise ValueError('Historical dataset must be populated')
    if 'product_family' in frame and (frame.product_family.isna().any() or
                                      frame.product_family.astype(str).str.strip().eq('').any()):
        raise ValueError('Provided dated product families must be populated')
    frame[target] = pd.to_numeric(frame[target], errors='raise')
    if not frame[target].dropna().isin([0, 1]).all():
        raise ValueError('Outcome labels must be binary or missing')
    for date in DATES:
        frame[date] = pd.to_datetime(frame[date], errors='raise', utc=True).dt.tz_localize(None)
    if frame[['observation_date', 'feature_as_of', 'label_window_end']].isna().any().any():
        raise ValueError('Snapshot, feature-as-of and outcome-window dates are required')
    observed = frame[target].notna()
    if frame.loc[observed, 'label_available_at'].isna().any():
        raise ValueError('Observed outcomes require label availability dates')
    if (frame.feature_as_of > frame.observation_date).any():
        raise ValueError('Features must be available by the observation timestamp')
    expected_end = frame.observation_date.map(lambda value: value + pd.DateOffset(years=1))
    if not frame.label_window_end.eq(expected_end).all():
        raise ValueError('Each outcome window must end exactly 12 months after observation')
    if (frame.loc[observed, 'label_available_at'] < frame.loc[observed, 'label_window_end']).any():
        raise ValueError('Labels cannot be declared available before their window matures')
    if frame.duplicated(['account_id', 'array_id', 'observation_date']).any():
        raise ValueError('Duplicate entity/snapshot keys')
    frame[TARGET] = frame[target]
    if target == 'capacity_expanded_next_12m':
        # Prevent an externally assigned expansion tag from becoming an outcome proxy.
        frame['expansion_potential_signal'] = 'UNKNOWN'
    validated = [validate_raw_frame(group) for _, group in frame.groupby('observation_date', sort=True)]
    return pd.concat(validated, ignore_index=True)


def chronological_split(frame, cutoff, evaluation_as_of):
    cutoff, evaluation_as_of = pd.Timestamp(cutoff), pd.Timestamp(evaluation_as_of)
    if cutoff.tzinfo or evaluation_as_of.tzinfo:
        raise ValueError('Use explicit UTC-naive cutoffs matching normalized input timestamps')
    if evaluation_as_of <= cutoff:
        raise ValueError('Evaluation-as-of must follow the training cutoff')
    labeled = frame[TARGET].notna()
    train = frame.loc[labeled & (frame.observation_date < cutoff) &
                      (frame.label_window_end < cutoff) & (frame.label_available_at < cutoff)].copy()
    # Latest fully matured snapshot per entity avoids duplicate training rows and long-history overweighting.
    train = train.sort_values('observation_date').drop_duplicates(['account_id', 'array_id'], keep='last')
    validation = frame.loc[labeled & frame.observation_date.eq(cutoff) &
                           (frame.label_window_end <= evaluation_as_of) &
                           (frame.label_available_at <= evaluation_as_of)].copy()
    if train.empty or validation.empty:
        raise ValueError('Cutoff needs earlier matured training labels and matured cutoff-snapshot validation labels')
    return train.reset_index(drop=True), validation.reset_index(drop=True)


def rolling_origin_backtest(data, cutoffs, evaluation_as_of, outcome_definition,
                            target=TARGET, entity_level='array', output_root=OUTPUTS,
                            inner_folds=3, params=None):
    if not isinstance(outcome_definition, str) or not outcome_definition.strip():
        raise ValueError('An explicit business outcome definition is required')
    frame = validate_history(data, target, entity_level)
    dates = [pd.Timestamp(value) for value in cutoffs]
    if not dates or len(set(dates)) != len(dates) or dates != sorted(dates):
        raise ValueError('Provide unique increasing cutoffs')
    # Validate all windows before producing a partial output directory.
    splits = [chronological_split(frame, date, evaluation_as_of) for date in dates]
    for train, _ in splits:
        if train.account_id.nunique() < inner_folds or train[TARGET].nunique() != 2:
            raise ValueError('Each training window needs enough accounts and both outcome classes')
    run_id = uuid.uuid4().hex
    root = Path(output_root) / 'temporal_runs' / run_id
    root.mkdir(parents=True, exist_ok=False)
    forecasts, metrics, windows, cohorts = [], [], [], []
    for fold, (date, (train, val)) in enumerate(zip(dates, splits), 1):
        reference = date.strftime('%Y-%m-%d')
        inner = inner_predictions(train, False, inner_folds, SEED + fold * 100,
                                  params, reference_date=reference)
        maps = fit_maps(inner, train[TARGET].to_numpy())
        pair = fit_pair(train, False, SEED + fold, params, reference_date=reference)
        raw = predict_pair(pair, val)
        scored = val[['account_id', 'array_id', 'observation_date', 'label_window_end',
                      'label_available_at', TARGET]].copy()
        scored['training_cutoff'] = date
        scored['run_id'] = run_id
        scored['prediction_source'] = 'chronological_holdout'
        if 'product_family' in val:
            scored['product_family'] = val.product_family
        scored['training_prevalence'] = float(train[TARGET].mean())
        columns = ['training_prevalence']
        if target != 'capacity_expanded_next_12m':
            scored['static_5'] = .05
            columns.append('static_5')
        for component, p in raw.items():
            name = f'{component}_sigmoid'
            scored[name] = maps[name].predict(p)
            columns.append(name)
        forecasts.append(scored)
        for model in columns:
            metrics.append({'training_cutoff': reference, 'model': model, 'rows': len(val),
                            **probability_metrics(val[TARGET].to_numpy(), scored[model].to_numpy())})
            if 'product_family' in scored:
                for family, group in scored.groupby('product_family'):
                    cohorts.append({'training_cutoff': reference, 'product_family': family,
                                    'model': model, 'rows': len(group),
                                    **probability_metrics(group[TARGET].to_numpy(), group[model].to_numpy())})
        windows.append({'cutoff': reference, 'reference_date': reference,
                        'train_rows': len(train), 'validation_rows': len(val),
                        'training_prevalence': float(train[TARGET].mean()),
                        'latest_training_label_available_at': str(train.label_available_at.max()),
                        'train_keys': train[['account_id', 'array_id', 'observation_date']].astype(str).to_dict('records'),
                        'validation_keys': val[['account_id', 'array_id', 'observation_date']].astype(str).to_dict('records')})
        with (root / f'window_{fold}_bundle.pkl').open('wb') as output:
            pickle.dump({'pair': pair, 'calibrators': maps, 'outcome_definition': outcome_definition,
                         'target': target, 'reference_date': reference}, output)
    predictions = pd.concat(forecasts, ignore_index=True)
    if entity_level == 'account':
        predictions = predictions.drop(columns='array_id')
    if target != TARGET:
        predictions = predictions.rename(columns={TARGET: target})
    predictions.to_csv(root / 'historical_predictions.csv', index=False)
    pd.DataFrame(metrics).to_csv(root / 'historical_metrics.csv', index=False)
    if cohorts:
        pd.DataFrame(cohorts).to_csv(root / 'historical_product_cohorts.csv', index=False)
    frame.to_csv(root / 'validated_history.csv', index=False)
    manifest = {
        'run_id': run_id, 'model_version': MODEL_VERSION, 'target': target, 'entity_level': entity_level,
        'python_version': platform.python_version(), 'seed': SEED, 'inner_folds': inner_folds,
        'parameters': {**DEFAULT_PARAMS, **(params or {})},
        'versions': {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'xgboost']},
        'outcome_definition': outcome_definition, 'evaluation_as_of': str(pd.Timestamp(evaluation_as_of)),
        'validation_design': 'rolling_origin_matured_labels_latest_training_snapshot_per_entity',
        'inner_calibration_design': 'account_grouped_training_oof', 'windows': windows,
        'account_overlap_policy': 'Existing accounts may recur in later windows; IDs are excluded from features',
        'feature_availability_policy': 'Input feature_as_of must not exceed observation_date; caller must supply genuine as-of values',
        'artifacts': {p.name: sha256_file(p) for p in root.iterdir() if p.is_file()},
        'source_sha256': {p.name: sha256_file(p) for p in Path(__file__).parent.glob('*.py')},
    }
    (root / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return root, predictions, pd.DataFrame(metrics)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data', type=Path, required=True)
    parser.add_argument('--cutoffs', required=True, help='Comma-separated observation timestamps')
    parser.add_argument('--evaluation-as-of', required=True)
    parser.add_argument('--outcome-definition', required=True)
    parser.add_argument('--target', choices=sorted(TARGETS), default=TARGET)
    parser.add_argument('--entity-level', choices=['array', 'account'], default='array')
    args = parser.parse_args()
    root, _, _ = rolling_origin_backtest(args.data, args.cutoffs.split(','), args.evaluation_as_of,
                                         args.outcome_definition, args.target, args.entity_level)
    print(f'Chronological evidence saved: {root}')


if __name__ == '__main__':
    main()
'''
write_embedded('src/temporal_validation.py', MODULE_CODE, [], '2a0a561aaa23ddbe78987a88d17f68e8297a3993d951a2d7a22bd60826350a17')

In [ ]:
# Embedded src/utils.py
MODULE_CODE = r'''"""Shared risk-bucket policy for Python validation and warehouse exports."""
import numpy as np


def assign_risk_bucket(prob: float) -> str:
    """Apply the fixed descriptive thresholds to a finite probability."""
    if not np.isfinite(prob) or not 0 <= prob <= 1:
        raise ValueError('Risk bucket requires a finite probability in [0, 1]')
    if prob >= 0.70:
        return 'CRITICAL'
    if prob >= 0.40:
        return 'HIGH'
    if prob >= 0.15:
        return 'MEDIUM'
    return 'LOW'
'''
write_embedded('src/utils.py', MODULE_CODE, [14], '6d4f4fe656f94e03f3d493348077a91f92357b3411a505f14655dd2915aa87ac')

In [ ]:
# Embedded src/verification.py
MODULE_CODE = r'''"""Read-only delivery checks; does not train, score, or generate reports."""
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
from src.config import ROOT, DATA_PATH, OUTPUTS, SCORE_COLUMNS
from src.data_cleaning import clean_dataset, validate_raw_frame, TARGET
from src.feature_engineering import CLVPreprocessor
from src.model_training import latest_run, load_bundle, frame_hash, predict_bundle, sha256_file
from src.backtesting import read_evidence
from src.utils import assign_risk_bucket
from src.contracts import validate_scored_output


def verify():
    run_dir = latest_run()
    evidence, manifest = read_evidence(run_dir)
    bundle, _ = load_bundle(run_dir)
    assert set(manifest['source_sha256']) == {p.name for p in (ROOT / 'src').glob('*.py')}
    assert set(manifest['warehouse_source_sha256']) == {p.name for p in (ROOT / 'sql').glob('*.sql')}
    for name, expected_hash in manifest.get('source_sha256', {}).items():
        assert sha256_file(ROOT / 'src' / name) == expected_hash, f'Source changed since training: {name}'
    for name, expected_hash in manifest['warehouse_source_sha256'].items():
        assert sha256_file(ROOT / 'sql' / name) == expected_hash, f'SQL changed since training: {name}'
    assert sha256_file(run_dir / 'training_snapshot.csv') == manifest['training_snapshot_sha256']
    raw = validate_raw_frame(DATA_PATH)
    labeled, unlabeled, audit = clean_dataset(raw)
    assert frame_hash(validate_raw_frame(labeled)) == manifest['labeled_input_sha256'], 'Training input hash mismatch'
    assert len(labeled) == len(evidence)
    assert audit['labeled_churn_count'] == int(evidence[TARGET].sum())
    assert np.array_equal(evidence[['account_id', 'array_id']].to_numpy(),
                          labeled[['account_id', 'array_id']].to_numpy())
    assert np.array_equal(evidence[TARGET].to_numpy(), labeled[TARGET].to_numpy())
    new = unlabeled.drop(columns=TARGET)
    assert np.isfinite(bundle['preprocessor'].transform(new)).all(), 'All-unlabeled inference failed'
    pointer = json.loads((OUTPUTS / 'latest_score_run.json').read_text())
    score_dir = OUTPUTS / pointer['directory']
    score_manifest = json.loads((score_dir / 'score_manifest.json').read_text())
    for name, field in [('CLV_SCORED_OUTPUT.csv', 'csv_sha256'), ('input_snapshot.csv', 'snapshot_sha256'),
                        ('xgb_explanations.json', 'explanations_sha256')]:
        assert sha256_file(score_dir / name) == score_manifest[field], f'{name} hash mismatch'
    scored = pd.read_csv(score_dir / 'CLV_SCORED_OUTPUT.csv', dtype={'account_id': str, 'array_id': str})
    validate_scored_output(scored)
    snapshot = validate_raw_frame(score_dir / 'input_snapshot.csv')
    assert score_manifest['training_run_id'] == manifest['run_id']
    assert list(scored.columns) == SCORE_COLUMNS and not scored.isna().any().any()
    assert len(scored) == len(snapshot) == score_manifest['rows']
    assert not scored.duplicated(['account_id', 'array_id', 'model_version']).any()
    assert scored.churn_risk_bucket.tolist() == scored.ensemble_churn_probability.map(assign_risk_bucket).tolist()
    p = predict_bundle(bundle, snapshot)
    for name in ('xgb', 'rf', 'ensemble'):
        np.testing.assert_allclose(scored[f'{name}_churn_probability'], p[name], atol=5.1e-7)
    analytics_dir = score_dir / 'submission_analytics'
    analytics = json.loads((analytics_dir / 'analytics_manifest.json').read_text())
    assert analytics['training_run_id'] == manifest['run_id']
    assert analytics['score_run_id'] == score_manifest['score_run_id']
    assert analytics['evaluation_sha256'] == manifest['evaluation_sha256']
    assert analytics['score_manifest_sha256'] == sha256_file(score_dir / 'score_manifest.json')
    for filename, metadata in analytics['artifacts'].items():
        assert sha256_file(analytics_dir / filename) == metadata['sha256'], filename
        assert len(pd.read_csv(analytics_dir / filename)) == metadata['rows'], filename
    accounts = pd.read_csv(analytics_dir / 'account_array_event_summary.csv')
    assert len(accounts) == snapshot.account_id.nunique()
    assert accounts.complete_probability_coverage.all()
    assert np.isclose(accounts.signed_historical_profit_usd.sum(), snapshot.historical_15yr_net_profit_usd.sum())
    horizons = pd.read_csv(analytics_dir / 'fifteen_year_scenarios.csv')
    assert len(horizons) == 2 * len(snapshot)
    for name in ['calibrated_array_event_rate', 'static_5pct']:
        group = horizons.loc[horizons.scenario.eq(name)]
        for year in range(1, 16):
            np.testing.assert_allclose(group[f'assumed_survival_year_{year}'],
                                       (1 - group.assumed_annual_event_probability)**year, atol=1e-14)
    report = run_dir / 'backtesting_report.md'
    assert report.exists() and 'grouped OOF' in report.read_text()
    assert sha256_file(report) == sha256_file(OUTPUTS / 'backtesting_report.md')
    metrics = pd.read_csv(run_dir / 'metrics.csv').set_index('model')
    y = evidence[TARGET].to_numpy()
    for model in metrics.index:
        assert np.isclose(metrics.loc[model, 'brier'], np.mean((evidence[model].to_numpy() - y)**2)), model
    ddl = (ROOT / 'sql/ddl_clv_scored_output.sql').read_text(encoding='utf-8-sig')
    body = ddl.split('CREATE TABLE IF NOT EXISTS CLV_SCORE_STAGING_TEMPLATE (')[1].split(');')[0]
    assert [line.strip().split()[0].lower() for line in body.splitlines() if line.strip()] == SCORE_COLUMNS
    assert 'CREATE TABLE IF NOT EXISTS CLV_SCORED_OUTPUT LIKE CLV_SCORED_HISTORY' in ddl
    assert 'CREATE OR REPLACE VIEW CLV_SCORED_OUTPUT' not in ddl
    handler = (ROOT / 'sql/stored_proc_scoring_pipeline.sql').read_text(encoding='utf-8-sig').split('$$')[1]
    compile(handler, 'snowflake_handler', 'exec')
    assert 'FROM @CLV_STAGE' in handler and 'TRUNCATE' not in handler
    assert 'MERGE INTO CLV_SCORED_OUTPUT' in handler and 'LATEST_MERGE_QUERY_ID' in handler
    for path in (ROOT / 'notebooks').glob('*.ipynb'):
        notebook = json.loads(path.read_text(encoding='utf-8'))
        assert not [o for c in notebook['cells'] for o in c.get('outputs', []) if o['output_type'] == 'error'], path
        assert all(c['execution_count'] is not None for c in notebook['cells'] if c['cell_type'] == 'code'), path
    print(f'PASS: input/evidence/bundle/output hashes, grouped folds, unlabeled inference, probabilities, '
          f'OOF metrics, account summaries, 15-year scenarios, CSV schema and executed notebooks. Run {manifest["run_id"]}')
    print('LIMITATION: SQL/Python static checks only; live Snowflake execution remains unverified.')
    return True

'''
write_embedded('src/verification.py', MODULE_CODE, [97], '7e619316603103e36e805cbc11af3a74031372b7e97052986758b5df55c79254')

### Snowflake scripts

The original five scripts are embedded for warehouse lineage and included in downloads. They are also supplied separately under `collab/sql/`. Creating these files does not execute Snowflake.

In [ ]:
# Embedded sql/ddl_clv_scored_output.sql
MODULE_CODE = r'''-- Standard table keys are informational. Use a single serialized ingestion job.
-- CSV shape is exactly 25 columns in this order.
CREATE TABLE IF NOT EXISTS CLV_SCORE_STAGING_TEMPLATE (
    ACCOUNT_ID VARCHAR(50) NOT NULL,
    ARRAY_ID VARCHAR(50) NOT NULL,
    CONTRACT_END_DATE DATE NOT NULL,
    HISTORICAL_15YR_NET_PROFIT_USD NUMBER(15,2) NOT NULL,
    CURRENT_CAPACITY_TB NUMBER(10,2) NOT NULL,
    UTILIZATION_RATE_PCT NUMBER(10,6) NOT NULL,
    RAW_UTILIZATION_RATE_PCT NUMBER(10,6) NOT NULL,
    UTILIZATION_CAPPED NUMBER(7,6) NOT NULL,
    IS_OVER_UTILIZED_RAW BOOLEAN NOT NULL,
    IS_NEGATIVE_PROFIT BOOLEAN NOT NULL,
    EXPANSION_POTENTIAL_SIGNAL VARCHAR(20) NOT NULL,
    EXPANSION_SIGNAL_ENCODED NUMBER(1,0) NOT NULL,
    STATIC_BASELINE_CHURN_RATE NUMBER(7,6) NOT NULL,
    XGB_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    RF_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    ENSEMBLE_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    CHURN_RISK_BUCKET VARCHAR(10) NOT NULL,
    SHAP_TOP_FEATURE_1 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_1_VALUE FLOAT NOT NULL,
    SHAP_TOP_FEATURE_2 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_2_VALUE FLOAT NOT NULL,
    SHAP_TOP_FEATURE_3 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_3_VALUE FLOAT NOT NULL,
    SCORED_AT TIMESTAMP_NTZ NOT NULL,
    MODEL_VERSION VARCHAR(50) NOT NULL
);
CREATE TABLE IF NOT EXISTS CLV_SCORED_HISTORY (
    ACCOUNT_ID VARCHAR(50) NOT NULL,
    ARRAY_ID VARCHAR(50) NOT NULL,
    CONTRACT_END_DATE DATE NOT NULL,
    HISTORICAL_15YR_NET_PROFIT_USD NUMBER(15,2) NOT NULL,
    CURRENT_CAPACITY_TB NUMBER(10,2) NOT NULL,
    UTILIZATION_RATE_PCT NUMBER(10,6) NOT NULL,
    RAW_UTILIZATION_RATE_PCT NUMBER(10,6) NOT NULL,
    UTILIZATION_CAPPED NUMBER(7,6) NOT NULL,
    IS_OVER_UTILIZED_RAW BOOLEAN NOT NULL,
    IS_NEGATIVE_PROFIT BOOLEAN NOT NULL,
    EXPANSION_POTENTIAL_SIGNAL VARCHAR(20) NOT NULL,
    EXPANSION_SIGNAL_ENCODED NUMBER(1,0) NOT NULL,
    STATIC_BASELINE_CHURN_RATE NUMBER(7,6) NOT NULL,
    XGB_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    RF_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    ENSEMBLE_CHURN_PROBABILITY NUMBER(7,6) NOT NULL,
    CHURN_RISK_BUCKET VARCHAR(10) NOT NULL,
    SHAP_TOP_FEATURE_1 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_1_VALUE FLOAT NOT NULL,
    SHAP_TOP_FEATURE_2 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_2_VALUE FLOAT NOT NULL,
    SHAP_TOP_FEATURE_3 VARCHAR(100) NOT NULL,
    SHAP_TOP_FEATURE_3_VALUE FLOAT NOT NULL,
    SCORED_AT TIMESTAMP_NTZ NOT NULL,
    MODEL_VERSION VARCHAR(50) NOT NULL,
    RUN_ID VARCHAR(32) NOT NULL,
    TRAINING_RUN_ID VARCHAR(32) NOT NULL,
    LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY (ACCOUNT_ID, ARRAY_ID, MODEL_VERSION, RUN_ID)
);
CREATE TABLE IF NOT EXISTS CLV_SCORE_RUNS (
    RUN_ID VARCHAR(32) NOT NULL,
    TRAINING_RUN_ID VARCHAR(32) NOT NULL,
    MODEL_VERSION VARCHAR(50) NOT NULL,
    FILE_NAME VARCHAR(500) NOT NULL,
    SNAPSHOT_SHA256 VARCHAR(64) NOT NULL,
    BUNDLE_SHA256 VARCHAR(64) NOT NULL,
    ROWS_LOADED INTEGER NOT NULL,
    COPY_QUERY_ID VARCHAR(100),
    MERGE_QUERY_ID VARCHAR(100),
    LATEST_MERGE_QUERY_ID VARCHAR(100),
    SHAP_UNITS VARCHAR(50) DEFAULT 'xgb_raw_margin_log_odds',
    COMPLETED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY (RUN_ID)
);
CREATE TABLE IF NOT EXISTS CLV_SCORE_FAILURES (
    RUN_ID VARCHAR(32), FAILED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    ERROR_MESSAGE VARCHAR(2000)
);
-- Physical current-score table required by the case-study deliverable.
-- Fresh sandbox installation. If an earlier version created a view under this
-- name, rename that view manually before installing this table; do not drop history.
CREATE TABLE IF NOT EXISTS CLV_SCORED_OUTPUT LIKE CLV_SCORED_HISTORY;
-- Independent reconciliation view; this is not the ingestion target.
CREATE OR REPLACE VIEW CLV_SCORED_LATEST_FROM_HISTORY AS
SELECT ACCOUNT_ID, ARRAY_ID, CONTRACT_END_DATE, HISTORICAL_15YR_NET_PROFIT_USD, CURRENT_CAPACITY_TB, UTILIZATION_RATE_PCT, RAW_UTILIZATION_RATE_PCT, UTILIZATION_CAPPED, IS_OVER_UTILIZED_RAW, IS_NEGATIVE_PROFIT, EXPANSION_POTENTIAL_SIGNAL, EXPANSION_SIGNAL_ENCODED, STATIC_BASELINE_CHURN_RATE, XGB_CHURN_PROBABILITY, RF_CHURN_PROBABILITY, ENSEMBLE_CHURN_PROBABILITY, CHURN_RISK_BUCKET, SHAP_TOP_FEATURE_1, SHAP_TOP_FEATURE_1_VALUE, SHAP_TOP_FEATURE_2, SHAP_TOP_FEATURE_2_VALUE, SHAP_TOP_FEATURE_3, SHAP_TOP_FEATURE_3_VALUE, SCORED_AT, MODEL_VERSION, RUN_ID, TRAINING_RUN_ID, LOADED_AT
FROM CLV_SCORED_HISTORY
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ACCOUNT_ID, ARRAY_ID, MODEL_VERSION
    ORDER BY SCORED_AT DESC, RUN_ID DESC
) = 1;

'''
write_embedded('sql/ddl_clv_scored_output.sql', MODULE_CODE, [91], '6637c3a1c17662fc352af03ee53425f88a3f7410c77452a9d825af369bf11adf')

In [ ]:
# Embedded sql/insert_scored_output.sql
MODULE_CODE = r'''-- Ingestion is performed by SP_INGEST_CLV_SCORED_BATCH, which validates before MERGE.
-- Never directly MERGE production scores without a unique run ID and checked staging.
-- Stage a unique <score_run_id>/CLV_SCORED_OUTPUT.csv using the README PUT command.
-- Then substitute exact score_manifest.json values into this six-argument insert call:
-- CALL SP_INGEST_CLV_SCORED_BATCH(
--   '<score_run_id>/CLV_SCORED_OUTPUT.csv', 'v2.1.0-grouped-sigmoid-unweighted',
--   '<score_run_id>', '<training_run_id>', '<snapshot_sha256>', '<bundle_sha256>'
-- );
-- The handler performs INSERT through MERGE into CLV_SCORED_OUTPUT and history,
-- then commits both tables and the ledger atomically. Retain all three query IDs.
-- Execute only in a sandbox after setting role/warehouse/database/schema.
-- Record all query IDs and procedure results. These instructions are not evidence of execution.
-- 1. Stage a unique fixture file; call the six-argument procedure with score_manifest.json values.
-- 2. Repeat identical call: expect REPLAY_UNCHANGED and unchanged history counts/payload.
-- 3. Stage changed scores under the same RUN_ID: expect rejection.
-- 4. New RUN_ID with later scores: expect new history rows and the current table updated.
-- 5. Stage duplicate keys, wrong version, invalid probability or bucket: expect rejection.
-- 6. Simulate failed COPY: history/ledger must remain unchanged; inspect failure log.
-- 7. Enforce one writer job; do not concurrently retry the same RUN_ID.
SELECT RUN_ID, COUNT(*) AS ROWS, COUNT(DISTINCT ACCOUNT_ID || ':' || ARRAY_ID || ':' || MODEL_VERSION) AS UNIQUE_KEYS
FROM CLV_SCORED_HISTORY GROUP BY RUN_ID;
SELECT RUN_ID, ACCOUNT_ID, ARRAY_ID, MODEL_VERSION, COUNT(*) AS N
FROM CLV_SCORED_HISTORY GROUP BY 1,2,3,4 HAVING COUNT(*) > 1;
SELECT RUN_ID, COUNT(*) AS N FROM CLV_SCORE_RUNS GROUP BY 1 HAVING COUNT(*) > 1;
SELECT * FROM CLV_SCORE_RUNS ORDER BY COMPLETED_AT DESC;
SELECT * FROM CLV_SCORE_FAILURES ORDER BY FAILED_AT DESC;
SELECT MODEL_VERSION, CHURN_RISK_BUCKET, COUNT(*) AS ARRAYS,
       AVG(ENSEMBLE_CHURN_PROBABILITY) AS MEAN_PROBABILITY,
       SUM(HISTORICAL_15YR_NET_PROFIT_USD) AS SIGNED_HISTORICAL_EXPOSURE_USD
FROM CLV_SCORED_OUTPUT GROUP BY 1,2;

'''
write_embedded('sql/insert_scored_output.sql', MODULE_CODE, [30], '3ce877a0933a7d09f1dda36051b249063ce11ed2bd92354d07c87e14fcce4b31')

In [ ]:
# Embedded sql/sandbox_acceptance.sql
MODULE_CODE = r'''-- Execute only in a sandbox after setting role/warehouse/database/schema.
-- Record all query IDs and procedure results. These instructions are not evidence of execution.
-- 1. Stage a unique fixture file; call the six-argument procedure with score_manifest.json values.
-- 2. Repeat identical call: expect REPLAY_UNCHANGED and unchanged history counts/payload.
-- 3. Stage changed scores under the same RUN_ID: expect rejection.
-- 4. New RUN_ID with later scores: expect new history rows and current table updated.
-- 5. Stage duplicate keys, wrong version, invalid probability or bucket: expect rejection.
-- 6. Simulate failed COPY: history/ledger must remain unchanged; inspect failure log.
-- 7. Enforce one writer job; do not concurrently retry the same RUN_ID.
-- 8. Load an older timestamp: history grows, current-score rows must not regress.
-- 9. Fail the current-table MERGE: history/current/ledger transaction must roll back.
-- 10. Check timestamp ties: lexically larger RUN_ID wins in both table and history view.
SELECT RUN_ID, COUNT(*) AS ROWS, COUNT(DISTINCT ACCOUNT_ID || ':' || ARRAY_ID || ':' || MODEL_VERSION) AS UNIQUE_KEYS
FROM CLV_SCORED_HISTORY GROUP BY RUN_ID;
SELECT RUN_ID, ACCOUNT_ID, ARRAY_ID, MODEL_VERSION, COUNT(*) AS N
FROM CLV_SCORED_HISTORY GROUP BY 1,2,3,4 HAVING COUNT(*) > 1;
SELECT RUN_ID, COUNT(*) AS N FROM CLV_SCORE_RUNS GROUP BY 1 HAVING COUNT(*) > 1;
SELECT ACCOUNT_ID, ARRAY_ID, MODEL_VERSION, COUNT(*) AS N
FROM CLV_SCORED_OUTPUT GROUP BY 1,2,3 HAVING COUNT(*) > 1;
-- Expect zero rows; exclude the load timestamp from business-payload comparison.
SELECT * EXCLUDE LOADED_AT FROM CLV_SCORED_OUTPUT
MINUS SELECT * EXCLUDE LOADED_AT FROM CLV_SCORED_LATEST_FROM_HISTORY;
SELECT * EXCLUDE LOADED_AT FROM CLV_SCORED_LATEST_FROM_HISTORY
MINUS SELECT * EXCLUDE LOADED_AT FROM CLV_SCORED_OUTPUT;
SELECT * FROM CLV_SCORE_RUNS ORDER BY COMPLETED_AT DESC;
SELECT * FROM CLV_SCORE_FAILURES ORDER BY FAILED_AT DESC;
SELECT MODEL_VERSION, CHURN_RISK_BUCKET, COUNT(*) AS ARRAYS,
       AVG(ENSEMBLE_CHURN_PROBABILITY) AS MEAN_PROBABILITY,
       SUM(HISTORICAL_15YR_NET_PROFIT_USD) AS SIGNED_HISTORICAL_EXPOSURE_USD
FROM CLV_SCORED_OUTPUT GROUP BY 1,2;

'''
write_embedded('sql/sandbox_acceptance.sql', MODULE_CODE, [30], '67d70941b145473f68a0f9969232297623b76ae0860f31c4eeb4c4cfa4dc5d5a')

In [ ]:
# Embedded sql/setup.sql
MODULE_CODE = r'''-- Select an existing sandbox role/warehouse/database/schema in your SQL client first.
-- Execute scripts in order: setup, ddl, stored procedure, sandbox_acceptance.
ALTER SESSION SET TIMEZONE = 'UTC';
CREATE FILE FORMAT IF NOT EXISTS CLV_SCORE_CSV
 TYPE = CSV COMPRESSION = AUTO SKIP_HEADER = 1
 FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ('', 'NULL', 'null', 'NaN')
 DATE_FORMAT = 'YYYY-MM-DD' TIMESTAMP_FORMAT = 'YYYY-MM-DD HH24:MI:SS'
 ERROR_ON_COLUMN_COUNT_MISMATCH = TRUE;
CREATE STAGE IF NOT EXISTS CLV_STAGE FILE_FORMAT = CLV_SCORE_CSV;

'''
write_embedded('sql/setup.sql', MODULE_CODE, [9], '9b75dcdca8e59fec6d2f5b42be4c5f504dc59fcede7412587a4eff1dd83c8710')

In [ ]:
# Embedded sql/stored_proc_scoring_pipeline.sql
MODULE_CODE = r'''-- Static design only until compiled and exercised in the user's Snowflake sandbox.
-- Per-call staging is isolated; the ingestion scheduler MUST serialize calls.
-- In particular, simultaneous retries of the same RUN_ID are not safe under informational keys.
CREATE OR REPLACE PROCEDURE SP_INGEST_CLV_SCORED_BATCH(
    FILE_NAME VARCHAR, TARGET_MODEL_VERSION VARCHAR, RUN_ID VARCHAR,
    TRAINING_RUN_ID VARCHAR, SNAPSHOT_SHA256 VARCHAR, BUNDLE_SHA256 VARCHAR
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'ingest'
EXECUTE AS CALLER
AS
$$
import re
import uuid

COLUMNS = ["ACCOUNT_ID","ARRAY_ID","CONTRACT_END_DATE","HISTORICAL_15YR_NET_PROFIT_USD","CURRENT_CAPACITY_TB","UTILIZATION_RATE_PCT","RAW_UTILIZATION_RATE_PCT","UTILIZATION_CAPPED","IS_OVER_UTILIZED_RAW","IS_NEGATIVE_PROFIT","EXPANSION_POTENTIAL_SIGNAL","EXPANSION_SIGNAL_ENCODED","STATIC_BASELINE_CHURN_RATE","XGB_CHURN_PROBABILITY","RF_CHURN_PROBABILITY","ENSEMBLE_CHURN_PROBABILITY","CHURN_RISK_BUCKET","SHAP_TOP_FEATURE_1","SHAP_TOP_FEATURE_1_VALUE","SHAP_TOP_FEATURE_2","SHAP_TOP_FEATURE_2_VALUE","SHAP_TOP_FEATURE_3","SHAP_TOP_FEATURE_3_VALUE","SCORED_AT","MODEL_VERSION"]

def ingest(session, file_name, target_model_version, run_id, training_run_id, snapshot_sha256, bundle_sha256):
    if not re.fullmatch(r"[A-Za-z0-9_./-]{1,500}", file_name or "") or ".." in file_name or file_name.startswith("/"):
        raise ValueError("Use a safe relative file name on the fixed CLV_STAGE")
    if not re.fullmatch(r"[A-Za-z0-9_.-]{1,50}", target_model_version or ""):
        raise ValueError("Invalid model version")
    for value in (run_id, training_run_id):
        if not re.fullmatch(r"[a-f0-9]{32}", value or ""):
            raise ValueError("Run IDs must be 32 lowercase hex characters")
    for value in (snapshot_sha256, bundle_sha256):
        if not re.fullmatch(r"[a-f0-9]{64}", value or ""):
            raise ValueError("Hashes must be 64 lowercase hex characters")

    staging = "TMP_CLV_" + uuid.uuid4().hex.upper()
    columns = ", ".join(COLUMNS)
    in_transaction = False
    try:
        session.sql("CREATE TEMPORARY TABLE " + staging + " LIKE CLV_SCORE_STAGING_TEMPLATE").collect()
        copy_sql = ("COPY INTO " + staging + " FROM @CLV_STAGE FILES = ('" + file_name +
                    "') FILE_FORMAT = (FORMAT_NAME = 'CLV_SCORE_CSV') ON_ERROR = 'ABORT_STATEMENT' FORCE = TRUE")
        copy_result = session.sql(copy_sql).collect()
        copy_query_id = session.sql("SELECT LAST_QUERY_ID()").collect()[0][0]
        rows = session.sql("SELECT COUNT(*) FROM " + staging).collect()[0][0]
        if rows == 0:
            raise ValueError("Empty or missing staged batch")
        duplicate_count = session.sql("SELECT COUNT(*) FROM (SELECT ACCOUNT_ID, ARRAY_ID, MODEL_VERSION FROM " +
            staging + " GROUP BY 1,2,3 HAVING COUNT(*) > 1)").collect()[0][0]
        if duplicate_count:
            raise ValueError("Duplicate scoring keys")
        invalid_sql = """
          SELECT COUNT(*) FROM {table}
          WHERE MODEL_VERSION <> ? OR
          CURRENT_CAPACITY_TB <= 0 OR UTILIZATION_RATE_PCT < 0 OR RAW_UTILIZATION_RATE_PCT < 0 OR
          UTILIZATION_CAPPED < 0 OR UTILIZATION_CAPPED > 1 OR
          ABS(UTILIZATION_RATE_PCT - RAW_UTILIZATION_RATE_PCT) > 0.000001 OR
          ABS(UTILIZATION_CAPPED - LEAST(RAW_UTILIZATION_RATE_PCT, 1)) > 0.000001 OR
          IS_OVER_UTILIZED_RAW <> (RAW_UTILIZATION_RATE_PCT > 1) OR
          IS_NEGATIVE_PROFIT <> (HISTORICAL_15YR_NET_PROFIT_USD < 0) OR
          ABS(STATIC_BASELINE_CHURN_RATE - 0.05) > 0.000001 OR
          XGB_CHURN_PROBABILITY < 0 OR XGB_CHURN_PROBABILITY > 1 OR
          RF_CHURN_PROBABILITY < 0 OR RF_CHURN_PROBABILITY > 1 OR
          ENSEMBLE_CHURN_PROBABILITY < 0 OR ENSEMBLE_CHURN_PROBABILITY > 1 OR
          EXPANSION_POTENTIAL_SIGNAL NOT IN ('UNKNOWN','Low','Medium','High') OR
          EXPANSION_SIGNAL_ENCODED <> CASE EXPANSION_POTENTIAL_SIGNAL
            WHEN 'UNKNOWN' THEN 0 WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2 WHEN 'High' THEN 3 END OR
          CHURN_RISK_BUCKET <> CASE
            WHEN ENSEMBLE_CHURN_PROBABILITY >= 0.70 THEN 'CRITICAL'
            WHEN ENSEMBLE_CHURN_PROBABILITY >= 0.40 THEN 'HIGH'
            WHEN ENSEMBLE_CHURN_PROBABILITY >= 0.15 THEN 'MEDIUM' ELSE 'LOW' END
        """.format(table=staging)
        if session.sql(invalid_sql, params=[target_model_version]).collect()[0][0]:
            raise ValueError("Batch failed model/feature/probability/bucket checks")
        existing = session.sql(
            "SELECT TRAINING_RUN_ID, MODEL_VERSION, FILE_NAME, SNAPSHOT_SHA256, BUNDLE_SHA256, ROWS_LOADED "
            "FROM CLV_SCORE_RUNS WHERE RUN_ID = ?", params=[run_id]).collect()
        if existing:
            if len(existing) != 1 or tuple(existing[0]) != (
                    training_run_id, target_model_version, file_name, snapshot_sha256, bundle_sha256, rows):
                raise ValueError("Run ID reuse with changed lineage or duplicate ledger entries")
            difference = session.sql(
                "SELECT COUNT(*) FROM ((SELECT " + columns + " FROM " + staging +
                " MINUS SELECT " + columns + " FROM CLV_SCORED_HISTORY WHERE RUN_ID = ?) UNION ALL "
                "(SELECT " + columns + " FROM CLV_SCORED_HISTORY WHERE RUN_ID = ? MINUS SELECT " +
                columns + " FROM " + staging + "))", params=[run_id, run_id]).collect()[0][0]
            history_count = session.sql("SELECT COUNT(*) FROM CLV_SCORED_HISTORY WHERE RUN_ID = ?", params=[run_id]).collect()[0][0]
            if difference or history_count != rows:
                raise ValueError("Run ID replay contains changed scores or inconsistent history")
            return {"status": "REPLAY_UNCHANGED", "run_id": run_id, "rows": rows, "copy_query_id": copy_query_id}
        session.sql("BEGIN TRANSACTION").collect()
        in_transaction = True
        source = "SELECT " + columns + ", ? AS RUN_ID, ? AS TRAINING_RUN_ID FROM " + staging
        merge = (
            "MERGE INTO CLV_SCORED_HISTORY AS t USING (" + source + ") AS s "
            "ON t.ACCOUNT_ID=s.ACCOUNT_ID AND t.ARRAY_ID=s.ARRAY_ID AND "
            "t.MODEL_VERSION=s.MODEL_VERSION AND t.RUN_ID=s.RUN_ID "
            "WHEN NOT MATCHED THEN INSERT (" + columns + ", RUN_ID, TRAINING_RUN_ID) VALUES (" +
            ", ".join("s." + c for c in COLUMNS) + ", s.RUN_ID, s.TRAINING_RUN_ID)")
        merge_result = session.sql(merge, params=[run_id, training_run_id]).collect()
        merge_query_id = session.sql("SELECT LAST_QUERY_ID()").collect()[0][0]
        inserted = int(merge_result[0][0])
        if inserted != rows:
            raise ValueError("Partial/duplicate run insertion; inspect serialized ingestion")
        # Timestamp ties use RUN_ID consistently with the independent history view.
        latest_merge = (
            "MERGE INTO CLV_SCORED_OUTPUT AS t USING (" + source + ") AS s "
            "ON t.ACCOUNT_ID=s.ACCOUNT_ID AND t.ARRAY_ID=s.ARRAY_ID AND t.MODEL_VERSION=s.MODEL_VERSION "
            "WHEN MATCHED AND (s.SCORED_AT > t.SCORED_AT OR "
            "(s.SCORED_AT = t.SCORED_AT AND s.RUN_ID > t.RUN_ID)) THEN UPDATE SET " +
            ", ".join("t." + c + "=s." + c for c in COLUMNS + ["RUN_ID", "TRAINING_RUN_ID"]) +
            ", t.LOADED_AT=CURRENT_TIMESTAMP() "
            "WHEN NOT MATCHED THEN INSERT (" + columns + ", RUN_ID, TRAINING_RUN_ID) VALUES (" +
            ", ".join("s." + c for c in COLUMNS) + ", s.RUN_ID, s.TRAINING_RUN_ID)")
        session.sql(latest_merge, params=[run_id, training_run_id]).collect()
        latest_merge_query_id = session.sql("SELECT LAST_QUERY_ID()").collect()[0][0]
        session.sql(
            "INSERT INTO CLV_SCORE_RUNS (RUN_ID, TRAINING_RUN_ID, MODEL_VERSION, FILE_NAME, "
            "SNAPSHOT_SHA256, BUNDLE_SHA256, ROWS_LOADED, COPY_QUERY_ID, MERGE_QUERY_ID, LATEST_MERGE_QUERY_ID) "
            "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            params=[run_id, training_run_id, target_model_version, file_name, snapshot_sha256,
                    bundle_sha256, rows, copy_query_id, merge_query_id, latest_merge_query_id]).collect()
        session.sql("COMMIT").collect()
        in_transaction = False
        return {"status": "SUCCESS", "run_id": run_id, "rows_loaded": rows,
                "rows_inserted": inserted, "copy_query_id": copy_query_id, "merge_query_id": merge_query_id,
                "latest_merge_query_id": latest_merge_query_id}
    except Exception as error:
        if in_transaction:
            session.sql("ROLLBACK").collect()
        session.sql("INSERT INTO CLV_SCORE_FAILURES (RUN_ID, ERROR_MESSAGE) VALUES (?, ?)",
                    params=[run_id, str(error)[:2000]]).collect()
        raise
    finally:
        session.sql("DROP TABLE IF EXISTS " + staging).collect()
$$;

'''
write_embedded('sql/stored_proc_scoring_pipeline.sql', MODULE_CODE, [133], '6ccf2ea68f90ed4a6d36268570c235929d13fa091662f5b590c76ada74121e65')

### Reference and frozen published artifacts

Fresh models are trained from your CSV. Their results are compared with the Windows submission, but sampled XGBoost retraining can differ across platforms. A separate frozen model/evidence payload is used only for the explicitly labeled **published replay** on matching input. That replay recalculates metrics from the original genuine OOF evidence and scores your CSV with the original saved model; it is not new training. Reproduction checks are strict for published replay, with tolerances 0.000002 for OOF/metrics/scored values and 0.00001 for full SHAP/account/scenario values. IDs, UTC times, hashes and plot bytes can differ. Other datasets skip published replay.

In [ ]:
# Numerical reference data, not pre-trained models. Training is still executed.
REFERENCE_ZIP_B64 = '...'
reference_bytes=base64.b64decode(REFERENCE_ZIP_B64)
assert hashlib.sha256(reference_bytes).hexdigest()=='26f9fc371514436e68a4714660cad0b778e16a56cf700c8ffe37f0469f48da73'
with zipfile.ZipFile(io.BytesIO(reference_bytes)) as archive:
    for name in archive.namelist():
        if '/' in name and name.split('/')[0]!='submission_analytics': raise ValueError('Unexpected reference path')
        path=PROJECT/'reference'/name
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(archive.read(name))
print('Frozen numerical reference:', 'c32ab4de1b813b7f555cfad7ef03bd8d56160ba1')


In [ ]:
# Frozen Windows training artifacts, explicitly used for published-result replay only.
published_bytes=base64.b64decode('...')
assert hashlib.sha256(published_bytes).hexdigest()=='a0b31346a223ea0cd0fcdcf3f57dffaaa077a9903423cbea15b27852f47065d0'
(PROJECT/'published_training.zip').write_bytes(published_bytes)
print('Frozen model/evidence payload prepared; it is not a fresh training result.')

## 3. Upload the original CSV

Select exactly one CSV. If it is already uploaded using the Colab Files pane, the cell discovers it. A CSV with a different filename is accepted and copied to the canonical runtime location. The pipeline retains input row order; do not sort or alter the case-study data when reproducing published outputs.

In [ ]:
data_override=os.environ.get('CLV_COLLAB_CSV')
if data_override:
    input_path=Path(data_override)
else:
    candidates=[p for p in Path('/content').glob('*.csv')] if Path('/content').is_dir() else []
    if len(candidates)==1:
        input_path=candidates[0]
    else:
        from google.colab import files
        uploaded=files.upload()
        names=[n for n in uploaded if n.lower().endswith('.csv')]
        if len(names)!=1 or len(uploaded)!=1: raise ValueError('Choose exactly one raw case-study CSV')
        input_path=Path(names[0])
raw_bytes=input_path.read_bytes()
(PROJECT/'data/Case_Study_2_CLV_Predictive_Analytics.csv').write_bytes(raw_bytes)
print('Uploaded:',input_path.name,'| bytes:',len(raw_bytes),'| SHA256:',hashlib.sha256(raw_bytes).hexdigest())

## 4. Audit and clean

Utilization is a fraction: 1.45 means 145%. Preserve raw values, cap the model feature at 1 and retain an anomaly flag. Keep negative profit and transform with `sign(profit) * log1p(abs(profit))`. UNKNOWN is a separate nominal category. Missing target labels are **never imputed** and do not enter model fitting, calibration or evaluation. Invalid schema, dates, capacity, categories and duplicate account/array keys fail explicitly.

In [ ]:
run_step(r'''
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.config import DATA_PATH
from src.data_cleaning import validate_raw_frame, clean_dataset, TARGET
raw=validate_raw_frame(DATA_PATH, require_target=True)
labeled,unlabeled,audit=clean_dataset(raw)
out=Path('outputs/cleaning')
labeled.to_csv(out/'cleaned_labeled.csv',index=False)
unlabeled.to_csv(out/'cleaned_unlabeled.csv',index=False)
pd.Series(audit,name='value').rename_axis('measure').to_csv(out/'data_quality_audit.csv')
(out/'audit.json').write_text(json.dumps(audit,indent=2))
assert len(labeled)+len(unlabeled)==len(raw)
assert labeled[TARGET].notna().all() and unlabeled[TARGET].isna().all()
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].bar(['No event','Event','Unlabeled'],[(labeled[TARGET]==0).sum(),(labeled[TARGET]==1).sum(),len(unlabeled)],color=['#17324d','#087e8b','#d59a33'])
axes[0].set(title='Target populations',ylabel='Arrays')
axes[1].hist(raw.utilization_rate_pct,bins=25,color='#17324d'); axes[1].axvline(1,color='#c4554d',ls='--')
axes[1].set(title='Utilization audit',xlabel='Raw utilization (fraction)')
axes[2].hist(labeled.signed_log_profit,bins=25,color='#087e8b')
axes[2].set(title='Signed profit transformation',xlabel='sign(profit) * log1p(abs(profit))')
fig.tight_layout(); fig.savefig(out/'cleaning_audit.png',dpi=150); plt.close(fig)
print(json.dumps(audit,indent=2))
''')

In [ ]:
show_csv(PROJECT/'outputs/cleaning/data_quality_audit.csv')
show_images(PROJECT/'outputs/cleaning',['cleaning_audit.png'])

## 5. Train with leakage controls

Five outer GroupKFold folds keep sibling arrays together. Each outer-training fold independently learns capacity quartiles and generates three inner grouped OOF folds for calibration. Sigmoid and isotonic maps never fit outer validation labels. All 15 features exclude IDs, target and static baseline. XGB uses depth 4, learning rate 0.05, subsample/colsample 0.8, alpha 0.1 and lambda 1. RF uses depth 8, minimum split/leaf 10/4 and sqrt features. Both class-weighted and unweighted variants are retained.

The logistic challenger uses fold-local scaling. Training prevalence uses training labels only. The inference candidate is prespecified: **unweighted sigmoid ensemble**, not a winner chosen on outer results. Final inference models use all labeled rows with grouped inner-OOF maps. This full-size run can take several minutes on a free CPU runtime; progress prints per outer fold.

In [ ]:
run_step(r'''
import json
import pandas as pd
from pathlib import Path
from src.config import DATA_PATH
from src.model_training import train_and_evaluate_models
from src.backtesting import probability_metrics, score_columns
result=train_and_evaluate_models(DATA_PATH, n_splits=5, inner_splits=3)
oof=result['evaluation']; manifest=result['manifest']; run_dir=result['run_dir']
(run_dir/'collab_environment.json').write_bytes(Path('environment/runtime.json').read_bytes())
assert not oof.duplicated(['account_id','array_id']).any()
assert oof.groupby('account_id').fold_id.nunique().eq(1).all()
for f in manifest['folds']:
    assert not set(f['train_accounts']) & set(f['validation_accounts'])
folds=pd.DataFrame([{'fold':f['fold_id'],'training_rows':f['train_rows'],
                    'validation_rows':f['validation_rows'],'training_prevalence':f['training_prevalence'],
                    'capacity_edges':str(f['capacity_edges'])} for f in manifest['folds']])
folds.to_csv(run_dir/'fold_summary.csv',index=False)
metrics=pd.DataFrame([{'model':c,**probability_metrics(oof.churn_event_next_12m,oof[c])} for c in score_columns(oof)])
metrics.to_csv(run_dir/'training_metrics.csv',index=False)
pd.DataFrame({'feature':result['bundle']['preprocessor'].feature_names_}).to_csv(run_dir/'feature_names.csv',index=False)
print('Saved full-size training run:',run_dir.name)
''')

In [ ]:
run_id=json.loads((PROJECT/'outputs/latest_run.json').read_text())['run_id']
run_dir=PROJECT/'outputs/runs'/run_id
show_csv(run_dir/'feature_names.csv')
show_csv(run_dir/'fold_summary.csv')
show_csv(run_dir/'training_metrics.csv')

## 6. Compare fresh training with the published reference

Compare all 21 candidates by account/array key. Fresh Linux XGBoost training can differ from Windows despite matching seeds and libraries, so differences are reported without being disguised as exact reproduction. They do not prevent downloading the fresh run. On matching input, separately activate the frozen published artifacts for strict replay in Notebook 2. IDs and UTC timestamps are expected to change.

In [ ]:
run_step(r'''
from collab_support import check_reference,activate_published_run
from src.model_training import latest_run
check_reference('training',latest_run())
activate_published_run()
''')

## 7. Download the training handoff

The ZIP contains your CSV, cleaning audit/tables, fresh OOF/bundle/manifest, the separately identified frozen published run on matching input, and complete Python/SQL. Save it before the runtime expires. In a fresh Notebook 2 runtime, upload **this ZIP only**; the original CSV is inside. A trusted bundle from your own Notebook 1 is required.

In [ ]:
training_zip=PROJECT/'CLV_TRAINING_ARTIFACTS.zip'
support.make_archive(training_zip,'clv_training',run_dir)
offer_download(training_zip)

## Interpretation boundary

The OOF results are grouped cross-sectional evaluation. The CSV has no observation dates or verified matured windows, so this is not chronological historical back-testing. Missing labels are inference cases, not a held-out accuracy test. Model ranking and calibration must be assessed against prevalence, not only the outdated 5% assumption. Notebook 2 performs uncertainty, proxy-cohort, SHAP, account, capacity and horizon analyses.